**PS4 — Hourly Anomaly Detection**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture  : CUBIC MARS AWS v6
Target        : ensemble_anomaly_flag (binary: 1 if 2+ of 3 active signals)
Grain         : one row per DEVICE_ID per hour
Signals       : S1=SPC 3-sigma  S2=Isolation Forest  S3=Tap rejection >5%
                S4=SolarWinds network [INACTIVE — telemetry not available in Gold]
                S5=Bus ops ratio [REMOVED 2026-06-12 — CTA_REAL_TIME_BUS_DATA dropped]
NOTE          : signal_bus_ops (S5) and signal_network (S4) always default to 0.
                Effective ensemble = 2-of-3 active signals (S1, S2, S3).
Models        : 3-signal ensemble voting + HDBSCAN clustering + supervised XGBoost
Feasibility   : 95%  (5% gap: bus device IP coverage incomplete in SolarWinds)

---

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3",
    "hdbscan>=0.8","optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","scipy",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile
from scipy import stats

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 4
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler
from sklearn.ensemble                import (IsolationForest,
                                             RandomForestClassifier,
                                             StackingClassifier)
from sklearn.linear_model            import LogisticRegression
from sklearn.pipeline                import Pipeline
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             average_precision_score,
                                             confusion_matrix,
                                             classification_report)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
import hdbscan

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ──────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps4_hourly"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3", region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps4-anomaly-detection"
MLFLOW_MODEL_NAME   = "ps4-anomaly-detection"
MODEL_REGISTRY_NAME = "chicago-ps4-anomaly-detection"
FS_GROUP_NAME       = "chicago-ps4-features"
ENDPOINT_NAME       = "chicago-ps4-anomaly-v1"
TARGET              = "ensemble_anomaly_flag"
SIGNAL_THRESHOLD    = 2   # 2-of-3 active signals (S4 network + S5 bus_ops inactive)

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")

In [ ]:
# ── CELL 3b : Checkpoint — load saved state (skip Cells 4-8 if found) ────────
# Saves: df, train/val/test_df, X/y splits, FEATURE_COLS, SIGNAL_COLS, DEV_COL
# Path : /home/sagemaker-user/PS4/checkpoints/  (EFS — survives instance stop)
# Usage: Run Cells 1-3-3b → if CHECKPOINT_LOADED=True → jump to Cell 11 (SPC)
#        Set FORCE_RELOAD=True to re-pull from S3 and rebuild signals/features.

import os, joblib, numpy as _np_ckpt, pandas as _pd_ckpt

CKPT_DIR      = "/home/sagemaker-user/PS4/checkpoints"
FORCE_RELOAD  = False
CHECKPOINT_LOADED = False

_ckpt_df   = os.path.join(CKPT_DIR, "df.parquet")
_ckpt_meta = os.path.join(CKPT_DIR, "meta.joblib")

if not FORCE_RELOAD and os.path.exists(_ckpt_df) and os.path.exists(_ckpt_meta):
    print("[CHECKPOINT] Loading from", CKPT_DIR)
    df       = _pd_ckpt.read_parquet(_ckpt_df)
    train_df = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "train_df.parquet"))
    val_df   = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "val_df.parquet"))
    test_df  = _pd_ckpt.read_parquet(os.path.join(CKPT_DIR, "test_df.parquet"))
    _meta            = joblib.load(_ckpt_meta)
    FEATURE_COLS     = _meta["FEATURE_COLS"]
    SIGNAL_COLS      = _meta["SIGNAL_COLS"]
    TARGET           = _meta["TARGET"]
    DEV_COL          = _meta["DEV_COL"]
    DEVICE_TYPES     = _meta["DEVICE_TYPES"]
    SCALE_POS_WEIGHT = _meta["SCALE_POS_WEIGHT"]
    spc_results      = _meta.get("spc_results", {})
    _arrays          = joblib.load(os.path.join(CKPT_DIR, "arrays.joblib"))
    X_train = _arrays["X_train"]; y_train = _arrays["y_train"]
    X_val   = _arrays["X_val"];   y_val   = _arrays["y_val"]
    X_test  = _arrays["X_test"];  y_test  = _arrays["y_test"]
    X_cv    = _arrays["X_cv"];    y_cv    = _arrays["y_cv"]
    CHECKPOINT_LOADED = True
    print(f"  df           : {df.shape}")
    print(f"  train / val / test : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
    print(f"  X_train      : {X_train.shape}  positives: {y_train.sum():,}")
    print(f"  FEATURE_COLS : {len(FEATURE_COLS)}  |  SIGNAL_COLS: {len(SIGNAL_COLS)}")
    print(f"  SCALE_POS_WEIGHT: {SCALE_POS_WEIGHT}")
    print()
    print("  CHECKPOINT LOADED — skip Cells 4-8.")
    print("  Jump to Cell 9 (Feature Store) or Cell 11 (SPC analysis).")
else:
    print("[CHECKPOINT] Not found or FORCE_RELOAD=True — running full pipeline (Cells 4-8).")
    print("  Checkpoint will be saved automatically at end of Cell 8.")


In [ ]:
# ── CELL 4 : Load Gold table (Parquet on S3) ──────────────────────────────────────────────────
print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

# Gold table uses "hour_bucket" for the hourly timestamp; rename to hour_dt for downstream compat
if "hour_dt" not in df.columns and "hour_bucket" in df.columns:
    df = df.rename(columns={"hour_bucket": "hour_dt"})
df["hour_dt"] = pd.to_datetime(df["hour_dt"])

print(f"Parquet source  : confirmed")
print(f"Shape           : {df.shape}")
print(f"Date range      : {df['hour_dt'].min()}  →  {df['hour_dt'].max()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"Anomaly rate  : {df[TARGET].mean():.2%}")
print(f"\nData source: {DATA_SOURCE}  |  Gold version: {GOLD_VERSION}")

In [ ]:
# ── CELL 5 : Build signal ensemble flags ─────────────────────────────────────
# Gold pre-computes 3 signals from 28-day rolling baselines (event_rate_anomaly,
# metric_anomaly, reject_rate_anomaly).  Map these to PS4 signal names, then add
# TVM hardware signals (BHU/CHU/printer/OOS) so TVMs can reach ensemble threshold.
#
# Confirmed from Databricks (2026-06-30):
#   bhu/chu/printer/oos all present in Gold for TVM rows (1,784,896 rows)
#   bhu_nonzero_pct = 7.34%  ->  p90 must be computed only from TVM non-zero rows
#   oos_nonzero_pct = 34.68% ->  "any OOS event" is too broad; use p75 of non-zero

# ── Compute daily_event_count for SPC visualisation (Cell 12) ─────────────
if "event_count_hour" in df.columns and "transit_day" in df.columns:
    df["daily_event_count"] = df.groupby(
        ["DEVICE_ID", "transit_day"])["event_count_hour"].transform("sum")

# ── Signal 1: SPC event-rate anomaly (Gold 28-day baseline) ──────────────
if "event_rate_anomaly" in df.columns:
    df["signal_spc"] = df["event_rate_anomaly"].fillna(0).astype(int)
elif ("event_count_hour" in df.columns and "baseline_mean_events" in df.columns
      and "baseline_stddev_events" in df.columns):
    df["signal_spc"] = (
        (df["event_count_hour"] - df["baseline_mean_events"]).abs()
        > 3 * df["baseline_stddev_events"].fillna(1)
    ).astype(int)
else:
    df["signal_spc"] = 0

# ── Signal 2: Isolation Forest (placeholder filled in Cell 13) ──────────
if "signal_isolation_forest" not in df.columns:
    df["signal_isolation_forest"] = 0

# ── Signal 3: Tap rejection rate > 5% (GATE/VALIDATOR only — 0 for TVM) ─
if "tap_reject_rate_daily" in df.columns:
    df["signal_tap_rejection"] = (df["tap_reject_rate_daily"] > 5.0).astype(int)
elif "reject_rate_anomaly" in df.columns:
    df["signal_tap_rejection"] = df["reject_rate_anomaly"].fillna(0).astype(int)
else:
    df["signal_tap_rejection"] = 0

# ── Signal 4: Transaction-time latency anomaly (Gold metric_anomaly) ───────
if "metric_anomaly" in df.columns:
    df["signal_metric"] = df["metric_anomaly"].fillna(0).astype(int)
elif ("metric_401_avg_ms_hour" in df.columns
      and "baseline_mean_metric401" in df.columns
      and "baseline_stddev_metric401" in df.columns):
    df["signal_metric"] = (
        (df["metric_401_avg_ms_hour"] - df["baseline_mean_metric401"]).abs()
        > 2 * df["baseline_stddev_metric401"].fillna(1)
    ).astype(int)
else:
    df["signal_metric"] = 0

# ── Signal 5: Revenue-zero flag ───────────────────────────────────────────
if "use_revenue_zero_flag" in df.columns:
    df["signal_revenue_zero"] = df["use_revenue_zero_flag"].fillna(0).astype(int)
else:
    df["signal_revenue_zero"] = 0

# ── Signal 6: TVM hardware subsystem anomaly (BHU + CHU + PRINTER events) ─
# Fires when the sum of BHU+CHU+printer events in an hour is >= p90 of
# non-zero TVM hours only.  Reason: GATE/VALIDATOR always have hw_sum=0
# so a global p90 would be 0, and the condition would never fire.
# Confirmed: bhu_nonzero_pct=7.34% for TVM in Gold (2026-06-30).
_hw_cols = [c for c in ["bhu_count_hour", "chu_count_hour", "printer_count_hour"]
            if c in df.columns]
if _hw_cols:
    df["_hw_sum"] = df[_hw_cols].fillna(0).sum(axis=1)
    DEV_COL_TMP = next((c for c in df.columns
                        if c in ["mars_device_category","device_type","device_category"]), None)
    if DEV_COL_TMP:
        _tvm_mask  = df[DEV_COL_TMP].str.strip().str.upper() == "TVM"
        _tvm_nonzero = df.loc[_tvm_mask & (df["_hw_sum"] > 0), "_hw_sum"]
    else:
        _tvm_nonzero = df.loc[df["_hw_sum"] > 0, "_hw_sum"]
    if len(_tvm_nonzero) >= 50:
        _hw_p90 = _tvm_nonzero.quantile(0.90)
        df["signal_tvm_hw"] = (df["_hw_sum"] >= _hw_p90).astype(int)
        print(f"signal_tvm_hw: p90 of non-zero TVM HW events = {_hw_p90:.1f}")
    else:
        df["signal_tvm_hw"] = (df["_hw_sum"] > 0).astype(int)
        print("signal_tvm_hw: <50 TVM non-zero rows, using >0 threshold")
    df.drop(columns=["_hw_sum"], inplace=True)
else:
    df["signal_tvm_hw"] = 0
    print("signal_tvm_hw: bhu/chu/printer columns not found in Gold")

# ── Signal 7: Out-of-service event spike ──────────────────────────────────
# oos_count_hour > 0 fires for 34.68% of TVM hours (too broad: routine OOS logging).
# Use p75 of non-zero TVM oos hours to catch genuine OOS avalanches.
# Confirmed: oos_nonzero_pct=34.68% for TVM in Gold (2026-06-30).
if "oos_count_hour" in df.columns:
    _oos_nonzero = df.loc[df["oos_count_hour"] > 0, "oos_count_hour"]
    if len(_oos_nonzero) >= 50:
        _oos_p75 = _oos_nonzero.quantile(0.75)
        df["signal_oos"] = (df["oos_count_hour"] >= _oos_p75).astype(int)
        print(f"signal_oos:    p75 of non-zero OOS events = {_oos_p75:.1f}")
    else:
        df["signal_oos"] = (df["oos_count_hour"] > 0).astype(int)
else:
    df["signal_oos"] = 0
    print("signal_oos: oos_count_hour column not found")

# ── Ensemble: 2-of-N active signals ──────────────────────────────────────
SIGNAL_COLS = ["signal_spc", "signal_isolation_forest", "signal_tap_rejection",
               "signal_metric", "signal_revenue_zero", "signal_tvm_hw", "signal_oos"]
SIGNAL_COLS = [c for c in SIGNAL_COLS if c in df.columns]

df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
if TARGET not in df.columns or df[TARGET].sum() == 0:
    SIGNAL_THRESHOLD = 2
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)

# ── Device column detection ────────────────────────────────────────────────
DEV_COL = next((c for c in df.columns
                if c in ["mars_device_category", "device_type", "device_category"]), None)
DEVICE_TYPES = ["TVM", "GATE", "VALIDATOR"]

# ── Print summary ──────────────────────────────────────────────────────────
import pandas as _pd_sig
_SRC_MAP = {
    "signal_spc":             "event_rate_anomaly (Gold 28d baseline)",
    "signal_isolation_forest":"IF score — filled Cell 13",
    "signal_tap_rejection":   "tap_reject_rate_daily > 5%  [GATE/VALIDATOR]",
    "signal_metric":          "metric_anomaly — latency spike  [GATE/VALIDATOR]",
    "signal_revenue_zero":    "use_revenue_zero_flag  [GATE/VALIDATOR]",
    "signal_tvm_hw":          "BHU+CHU+printer >= p90 of non-zero TVM hrs  [TVM]",
    "signal_oos":             "oos_count_hour >= p75 of non-zero OOS hrs  [all]",
}
_rows = []
for s in SIGNAL_COLS:
    _row = {"Signal": s, "Source": _SRC_MAP.get(s, s),
            "Rate_All": f"{df[s].mean():.2%}",
            "Active_Rows": int(df[s].sum())}
    if DEV_COL:
        for _dt in DEVICE_TYPES:
            _m = df[df[DEV_COL].str.strip().str.upper()==_dt][s].mean()
            _row[f"Rate_{_dt}"] = f"{_m:.2%}"
    _rows.append(_row)
_rows.append({
    "Signal": TARGET, "Source": "ensemble (2+ signals)",
    "Rate_All": f"{df[TARGET].mean():.2%}",
    "Active_Rows": int(df[TARGET].sum()),
    **({f"Rate_{_dt}": f"{df[df[DEV_COL].str.strip().str.upper()==_dt][TARGET].mean():.2%}"
        for _dt in DEVICE_TYPES} if DEV_COL else {}),
})
print("\nSignal Summary:")
print(_pd_sig.DataFrame(_rows).to_string(index=False))

_neg = (df[TARGET]==0).sum(); _pos = max((df[TARGET]==1).sum(), 1)
print(f"\nscale_pos_weight hint: {round(_neg / _pos, 2)}")
if DEV_COL:
    print("")
    print(df.groupby(DEV_COL)[TARGET].agg(["count","mean"])
            .rename(columns={"count":"rows","mean":"anomaly_rate"}).to_string())


In [ ]:
# ── CELL 6 : Signal activation charts ──────────────────────────────────────────────────
import pandas as _pdv1

_sig_rates = {s: float(df[s].mean()) for s in SIGNAL_COLS if s in df.columns}
_sig_rates[TARGET] = float(df[TARGET].mean())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Horizontal bar: activation rate per signal ─────────────────────────
_sr = _pdv1.Series(_sig_rates).sort_values()
_colors = ["#E53935" if k == TARGET else "#1565C0" for k in _sr.index]
_sr.plot(kind="barh", ax=axes[0], color=_colors, edgecolor="white")
axes[0].axvline(0.05, color="orange", ls="--", lw=1.2, label="5% ref")
for bar, v in zip(axes[0].patches, _sr.values):
    axes[0].text(v + 0.001, bar.get_y() + bar.get_height()/2,
                 f"{v:.2%}", va="center", fontsize=8)
axes[0].set_title("Signal activation rates (all rows)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Activation rate"); axes[0].legend()

# ── 2. Device × signal heatmap ─────────────────────────────────────────────
if DEV_COL and DEV_COL in df.columns:
    _hmap = df.groupby(DEV_COL)[SIGNAL_COLS + [TARGET]].mean()
    sns.heatmap(_hmap.T, ax=axes[1], cmap="YlOrRd", annot=True, fmt=".1%",
                linewidths=0.5, cbar_kws={"label": "rate"})
    axes[1].set_title("Signal rate by device type", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Device type"); axes[1].set_ylabel("")
else:
    axes[1].text(0.5, 0.5, "No device column", ha="center", va="center",
                 transform=axes[1].transAxes)
    axes[1].set_title("Device × signal heatmap")

# ── 3. Signal co-activation matrix ─────────────────────────────────────────
_active = df[SIGNAL_COLS].astype(float)
_co = _active.T.dot(_active)  # count of rows where both signals are active
_co_pct = _co.div(_co.values.diagonal(), axis=0)  # normalise by row signal count
np.fill_diagonal(_co_pct.values, np.nan)
sns.heatmap(_co_pct, ax=axes[2], cmap="Blues", annot=True, fmt=".0%",
            linewidths=0.5, mask=_co_pct.isna(),
            cbar_kws={"label":"% of row signal rows where col also fires"})
axes[2].set_title("Signal co-activation matrix", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/ps4_signal_charts.png", dpi=100); plt.show()
with mlflow.start_run(run_name="signal_activation_viz") as _r:
    mlflow.set_tags({**globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag","city":"chicago"}), "model_type":"visualization"})
    mlflow.log_artifact("/tmp/ps4_signal_charts.png")

# ── Summary table ─────────────────────────────────────────────────────────
print("\n── Signal Summary Table ──")
_rows = []
for s in SIGNAL_COLS + [TARGET]:
    if s not in df.columns: continue
    _col = df[s]
    _row = {"Signal": s, "Rate_All": f"{_col.mean():.2%}",
            "Active_Rows": int(_col.sum())}
    if DEV_COL and DEV_COL in df.columns:
        for _dt in DEVICE_TYPES:
            _m = df[df[DEV_COL].str.strip().str.upper()==_dt][s].mean()
            _row[f"Rate_{_dt}"] = f"{_m:.2%}"
    _rows.append(_row)
print(_pdv1.DataFrame(_rows).to_string(index=False))


In [ ]:
# ── CELL 7 : EDA ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Signal co-activation heatmap
if len(SIGNAL_COLS) >= 2:
    sns.heatmap(df[SIGNAL_COLS].corr(), annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, ax=axes[0,0], linewidths=0.5)
    axes[0,0].set_title("Signal Co-activation Correlation")

# Anomaly flag distribution
df[TARGET].value_counts().plot(kind="bar", ax=axes[0,1],
    color=["#43A047","#E53935"], edgecolor="white")
axes[0,1].set_title("ensemble_anomaly_flag distribution")

# ── Temporal: hour-of-day × day-of-week heatmap ───────────────────────────
_hod = df["hour_dt"].dt.hour
_dow = df["hour_dt"].dt.day_name()
_DOW_ORDER = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
_pivot = (df.assign(hour=_hod, dow=_dow)
            .groupby(["dow","hour"])[TARGET].mean()
            .unstack(fill_value=0))
_pivot = _pivot.reindex([d for d in _DOW_ORDER if d in _pivot.index])
sns.heatmap(_pivot, ax=axes[0,2], cmap="YlOrRd", fmt=".0%",
            annot=True, linewidths=0.3, cbar_kws={"label":"anomaly rate"})
axes[0,2].set_title("Anomaly rate: hour × day-of-week")
axes[0,2].set_xlabel("Hour of day"); axes[0,2].set_ylabel("")

# ── Monthly anomaly rate trend ─────────────────────────────────────────────
_monthly = (df.assign(month=df["hour_dt"].dt.to_period("M"))
              .groupby("month")[TARGET].mean())
_monthly.index = _monthly.index.astype(str)
axes[1,0].plot(_monthly.index, _monthly.values, marker="o", color="#1565C0", lw=2)
axes[1,0].set_title("Monthly anomaly rate trend")
axes[1,0].set_xlabel("Month"); axes[1,0].set_ylabel("Anomaly rate")
axes[1,0].tick_params(axis="x", rotation=45)
axes[1,0].axhline(_monthly.mean(), color="red", ls="--", lw=1, label="mean")
axes[1,0].legend()

# Hourly pattern
df.groupby(_hod)[TARGET].mean().plot(kind="bar", ax=axes[1,1], color="#1565C0")
axes[1,1].set_title("Anomaly rate by hour of day")

# By device type
if DEV_COL and DEV_COL in df.columns:
    df.groupby(DEV_COL)[TARGET].mean().sort_values(ascending=False).plot(
        kind="bar", ax=axes[1,2], color="#7B1FA2")
    axes[1,2].set_title("Anomaly rate by device type")
else:
    axes[1,2].text(0.5, 0.5, "No device column", ha="center", va="center",
                   transform=axes[1,2].transAxes)

plt.tight_layout()
plt.savefig("/tmp/ps4_eda.png", dpi=100); plt.show()


In [ ]:
# ── CELL 8 : Feature engineering & splits ──────────────────────────────────────────────────
ID_COLS = ["DEVICE_ID","hour_dt","split"] + SIGNAL_COLS + ["signal_active_count"]
ID_COLS = [c for c in ID_COLS if c in df.columns]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [TARGET]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
print(f"Features: {len(FEATURE_COLS)}")

train_df = df[df["hour_dt"] <  "2025-10-01"]
val_df   = df[(df["hour_dt"] >= "2025-10-01") & (df["hour_dt"] < "2026-01-01")]
test_df  = df[df["hour_dt"] >= "2026-01-01"]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[TARGET].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[TARGET].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[TARGET].values
X_cv             = np.vstack([X_train,X_val])
y_cv             = np.concatenate([y_train,y_val])

neg, pos         = (y_train==0).sum(), (y_train==1).sum()
SCALE_POS_WEIGHT = round(neg/pos, 2)
print(f"Train:{len(X_train):,}  Val:{len(X_val):,}  Test:{len(X_test):,}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

# ── Save checkpoint after feature engineering & splits ───────────────────
import os, joblib as _jl
_ckpt_dir = "/home/sagemaker-user/PS4/checkpoints"
os.makedirs(_ckpt_dir, exist_ok=True)
df.to_parquet(       os.path.join(_ckpt_dir, "df.parquet"),       index=True)
train_df.to_parquet( os.path.join(_ckpt_dir, "train_df.parquet"), index=True)
val_df.to_parquet(   os.path.join(_ckpt_dir, "val_df.parquet"),   index=True)
test_df.to_parquet(  os.path.join(_ckpt_dir, "test_df.parquet"),  index=True)
_jl.dump({
    "FEATURE_COLS":     FEATURE_COLS,
    "SIGNAL_COLS":      SIGNAL_COLS,
    "TARGET":           TARGET,
    "DEV_COL":          DEV_COL,
    "DEVICE_TYPES":     DEVICE_TYPES,
    "SCALE_POS_WEIGHT": SCALE_POS_WEIGHT,
    "spc_results":      spc_results if "spc_results" in dir() else {},
}, os.path.join(_ckpt_dir, "meta.joblib"))
_jl.dump({
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
    "X_cv":    X_cv,    "y_cv":    y_cv,
}, os.path.join(_ckpt_dir, "arrays.joblib"))
print(f"[CHECKPOINT] Saved to {_ckpt_dir}")
print(f"  df={df.shape}  X_train={X_train.shape}  positives={y_train.sum():,}")
print(f"  Next kernel restart: run Cells 1-3-3b only, then jump to Cell 11.")


In [ ]:
# ── CELL 9 : Feature Store ──────────────────────────────────────────────────
# NOTE: This is the "Cell 9" seen in Jupyter UI (markdown cell at [0] shifts count).
# Set ENABLE_FEATURE_STORE=False to skip entirely during dev reruns — saves ~5 min.
ENABLE_FEATURE_STORE = False

if not ENABLE_FEATURE_STORE:
    print("[SKIP] Feature Store disabled — set ENABLE_FEATURE_STORE=True to enable")
else:
    # Ingest last 90 days of training data — representative for online serving,
    # avoids ingesting millions of rows from years of history.
    _cutoff_fs = train_df["hour_dt"].max() - pd.Timedelta(days=90)
    fs_df = train_df[train_df["hour_dt"] >= _cutoff_fs][
        FEATURE_COLS + [TARGET, "DEVICE_ID", "hour_dt"]].copy()
    fs_df["event_time"] = fs_df["hour_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
    fs_df               = fs_df.drop(columns=["hour_dt"])
    print(f"Feature Store sample: {len(fs_df):,} rows (last 90 days of train)")

    def dtype_to_fs(dtype):
        if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
        if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
        return FeatureTypeEnum.STRING

    feature_group = FeatureGroup(name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
        feature_definitions=[
            FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
            for c in fs_df.columns])

    _sm_fg_client = SM_SESSION.boto_session.client("sagemaker", region_name=REGION)
    _deleted = False
    try:
        _sm_fg_client.delete_feature_group(FeatureGroupName=FS_GROUP_NAME)
        print(f"Deleted existing Feature Group: {FS_GROUP_NAME}")
        _deleted = True
    except Exception:
        pass  # did not exist — fine

    if _deleted:
        print("Waiting 60s for deletion to propagate ...")
        time.sleep(60)   # only sleep when we actually deleted something

    try:
        feature_group.create(
            s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
            record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
            role_arn=ROLE, enable_online_store=True,
            description="PS4 hourly anomaly features — Chicago Ventra")
        _status = ""
        while _status != "Created":
            time.sleep(10)  # 10s instead of 15s
            _status = feature_group.describe().get("FeatureGroupStatus")
            print(f"  Feature Group status: {_status}")
    except Exception as _fg_e:
        print(f"Feature Group create: {_fg_e}")

    feature_group.ingest(
        data_frame=fs_df.reset_index(drop=True),
        max_workers=8,   # doubled from 4
        wait=True)
    print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")


In [ ]:
# ── CELL 10 : Helpers ──────────────────────────────────────────────────
COMMON_TAGS={"ps":"PS4","target":"ensemble_anomaly_flag","city":"chicago",
             "gold_version":str(GOLD_VERSION),"architecture":"CUBIC-MARS-AWS-v6"}

def evaluate(model,X,y,split="test"):
    prob=model.predict_proba(X)[:,1]; pred=(prob>=0.5).astype(int)
    return {f"{split}_auc" :round(roc_auc_score(y,prob),4),
            f"{split}_ap"  :round(average_precision_score(y,prob),4),
            f"{split}_f1"  :round(f1_score(y,pred),4)}

def log_cm(model,X,y,name):
    pred=(model.predict_proba(X)[:,1]>=0.5).astype(int)
    cm=confusion_matrix(y,pred)
    fig,ax=plt.subplots(figsize=(4,3))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=["Normal","Anomaly"],
                yticklabels=["Normal","Anomaly"])
    ax.set_title(name); fig.tight_layout()
    fig.savefig(f"/tmp/cm_{name}.png",dpi=80); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

In [ ]:
# ── CELL 11 : Signal 1 — SPC 3-sigma control chart (logged as analysis) ───────────────── ──────────
# Computes per-device UCL/LCL from daily_event_count, logs thresholds to MLflow.
# Note: daily_event_count was derived in Cell 5 from event_count_hour + transit_day.

spc_results = {}
if "daily_event_count" in df.columns:
    # Determine grouping: per-device-type if DEV_COL exists, else "ALL"
    _dev_groups = ([(dt, df[df[DEV_COL].str.strip().str.upper()==dt])
                    for dt in DEVICE_TYPES]
                   if DEV_COL else [("ALL", df)])
    for _dt_name, _subset in _dev_groups:
        _s = _subset["daily_event_count"].dropna()
        if len(_s) < 50:
            continue
        _mean, _std = _s.mean(), _s.std()
        spc_results[_dt_name] = {
            "mean": round(_mean, 4),
            "ucl":  round(_mean + 3 * _std, 4),
            "lcl":  max(0, round(_mean - 3 * _std, 4)),
        }
elif "event_rate_anomaly" in df.columns and DEV_COL:
    # Fallback: derive UCL/LCL from baseline columns directly
    for _dt_name in DEVICE_TYPES:
        _s = df[df[DEV_COL].str.strip().str.upper()==_dt_name]
        if len(_s) < 50: continue
        _mean = _s["baseline_mean_events"].mean() if "baseline_mean_events" in df.columns else 0
        _std  = _s["baseline_stddev_events"].mean() if "baseline_stddev_events" in df.columns else 1
        spc_results[_dt_name] = {
            "mean": round(_mean, 4),
            "ucl":  round(_mean + 3 * _std, 4),
            "lcl":  max(0, round(_mean - 3 * _std, 4)),
        }
else:
    print("[INFO] daily_event_count not found AND no baseline columns — SPC thresholds skipped.")

with mlflow.start_run(run_name="spc_signal1_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "spc_control_chart", "signal": "S1"})
    for dt_name, vals in spc_results.items():
        mlflow.log_metric(f"{dt_name}_mean", vals["mean"])
        mlflow.log_metric(f"{dt_name}_ucl",  vals["ucl"])
        mlflow.log_metric(f"{dt_name}_lcl",  vals["lcl"])
    with open("/tmp/spc_thresholds.json", "w") as f:
        json.dump(spc_results, f, indent=2)
    mlflow.log_artifact("/tmp/spc_thresholds.json")
    mlflow.log_param("sigma_threshold",     3)
    mlflow.log_param("rolling_window_hours", 28 * 24)
    mlflow.log_param("signal_spc_source",
                     "event_rate_anomaly (Gold)" if "event_rate_anomaly" in df.columns
                     else "daily_event_count (derived)")
    SPC_RUN = run.info.run_id

print(f"SPC thresholds logged for {len(spc_results)} device type(s): {list(spc_results.keys())}")
if spc_results:
    import pandas as _pdsp
    print(_pdsp.DataFrame(spc_results).T.to_string())


In [ ]:
# ── CELL 12 : SPC control charts + signal timeline ──────────────────────────
# Runtime note: raw hourly data has 30M+ rows per device type — unplottable.
# We resample to daily grain before plotting (365 points vs 10M+).
# SPC violations are shown as daily counts, not individual hourly scatter.
import pandas as _pdv2

# ── Per-device SPC control chart (daily-resampled) ────────────────────────
if "daily_event_count" in df.columns and DEV_COL and spc_results:
    _devs = [d for d in DEVICE_TYPES if d in spc_results]
    if _devs and "hour_dt" in df.columns:
        fig, axes = plt.subplots(len(_devs), 1, figsize=(16, 4*len(_devs)))
        if len(_devs) == 1: axes = [axes]
        for ax, _dt in zip(axes, _devs):
            _sub = (df[df[DEV_COL].str.strip().str.upper() == _dt]
                    .copy()
                    .set_index("hour_dt")
                    .sort_index())
            # Resample to daily mean — reduces 10M+ hourly points to ~365
            _daily = _sub["daily_event_count"].resample("1D").mean().dropna()
            _viol_daily = _sub["signal_spc"].resample("1D").sum().dropna()

            ax.plot(_daily.index, _daily.values,
                    lw=0.9, alpha=0.8, color="#1565C0", label="Daily mean event count")
            _ucl = spc_results[_dt]["ucl"]
            _lcl = spc_results[_dt]["lcl"]
            _mn  = spc_results[_dt]["mean"]
            ax.axhline(_ucl, color="red",    ls="--", lw=1.5, label=f"UCL = {_ucl:.0f}")
            ax.axhline(_lcl, color="#1976D2", ls="--", lw=1.5, label=f"LCL = {_lcl:.0f}")
            ax.axhline(_mn,  color="green",  ls="-",  lw=1.5, label=f"Mean = {_mn:.0f}")
            # Overlay daily SPC violation count as bar (right y-axis)
            ax2 = ax.twinx()
            ax2.bar(_viol_daily.index, _viol_daily.values,
                    width=0.8, alpha=0.3, color="red", label="SPC violation hrs/day")
            ax2.set_ylabel("Violation hours / day", color="red", fontsize=8)
            ax2.tick_params(axis="y", labelcolor="red", labelsize=7)
            ax.set_title(f"SPC 3-sigma Control Chart — {_dt} (daily grain)",
                         fontsize=11, fontweight="bold")
            ax.set_ylabel("Daily mean event count")
            # Merge legends
            _h1, _l1 = ax.get_legend_handles_labels()
            _h2, _l2 = ax2.get_legend_handles_labels()
            ax.legend(_h1 + _h2, _l1 + _l2, fontsize=7, ncol=2)
        plt.tight_layout()
        plt.savefig("/tmp/ps4_spc_chart.png", dpi=100); plt.show()
        _ct = globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag"})
        with mlflow.start_run(run_name="spc_control_chart") as _r:
            mlflow.set_tags({**_ct, "model_type": "visualization"})
            mlflow.log_artifact("/tmp/ps4_spc_chart.png")
    else:
        print("[INFO] spc_results empty or no hour_dt — SPC chart skipped.")
else:
    print("[INFO] daily_event_count or DEV_COL not available — SPC chart skipped.")

# ── Signal activation timeline (7-day rolling, daily grain) ───────────────
_time_sigs = [s for s in SIGNAL_COLS + [TARGET] if s in df.columns]
if "hour_dt" in df.columns and len(_time_sigs):
    _ts = (df.set_index("hour_dt")[_time_sigs]
             .sort_index()
             .resample("1D").mean()
             .rolling(7, min_periods=1).mean())
    fig, ax = plt.subplots(figsize=(16, 5))
    _COLORS = ["#1565C0","#E65100","#2E7D32","#6A1B9A","#00838F","#C62828","#F57F17"]
    for idx, s in enumerate(_time_sigs):
        _lw = 2.5 if s == TARGET else 1.2
        _ls = "-"  if s == TARGET else "--"
        ax.plot(_ts.index, _ts[s], label=s, lw=_lw, ls=_ls,
                color=_COLORS[idx % len(_COLORS)])
    ax.set_title("Signal activation rate — 7-day rolling average",
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Activation rate")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
    ax.legend(ncol=2, fontsize=8); plt.tight_layout()
    plt.savefig("/tmp/ps4_signal_timeline.png", dpi=100); plt.show()
    _ct = globals().get("COMMON_TAGS", {"ps":"PS4","target":"ensemble_anomaly_flag"})
    with mlflow.start_run(run_name="signal_timeline") as _r:
        mlflow.set_tags({**_ct, "model_type": "visualization"})
        mlflow.log_artifact("/tmp/ps4_signal_timeline.png")

# ── SPC threshold table ────────────────────────────────────────────────────
if spc_results:
    print("\n── SPC Thresholds ──")
    print(_pdv2.DataFrame(spc_results).T.to_string())
else:
    print("[INFO] SPC thresholds empty — daily_event_count signal did not fire.")


In [ ]:
# ── CELL 13 : Signal 2 — Isolation Forest (PCA-reduced, per-device contamination) ──────────────────────────────────────────────────
# PCA(95%) before IF: reduces 50+ features to ~15 components so IF trees
# sample the right dimensions instead of random noise columns.
# Per-device contamination: avoids one-size-fits-all 5% hardcoded rate.

IF_FEATURES = [c for c in FEATURE_COLS
               if any(k in c.lower() for k in
                      ["metric","response","percent_loss","kpi","rate","count"])]
IF_FEATURES = IF_FEATURES if IF_FEATURES else FEATURE_COLS[:15]

scaler = StandardScaler()
X_if_tr = scaler.fit_transform(train_df[IF_FEATURES].values)
X_if_va = scaler.transform(val_df[IF_FEATURES].values)
X_if_te = scaler.transform(test_df[IF_FEATURES].values)

# ── PCA: reduce before IF (fixes curse-of-dimensionality for isolation trees) ─
from sklearn.decomposition import PCA as _PCA
_pca = _PCA(n_components=0.95, random_state=42)
X_if_tr_pca = _pca.fit_transform(X_if_tr)
X_if_va_pca = _pca.transform(X_if_va)
X_if_te_pca = _pca.transform(X_if_te)
print(f"PCA: {len(IF_FEATURES)} features → {_pca.n_components_} components (95% variance)")

# ── Per-device contamination from actual observed anomaly rate ─────────────
_global_contam = float(np.clip(y_train.mean(), 0.01, 0.40))
if DEV_COL and DEV_COL in train_df.columns:
    _dev_contam = {}
    for _dt in DEVICE_TYPES:
        _mask = train_df[DEV_COL].str.strip().str.upper() == _dt
        _rate = float(train_df[_mask][TARGET].mean()) if _mask.sum() > 0 else _global_contam
        _dev_contam[_dt] = round(np.clip(_rate, 0.01, 0.40), 4)
    print(f"Per-device contamination: {_dev_contam}")
else:
    _dev_contam = {}

with mlflow.start_run(run_name="isolation_forest_signal2") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "isolation_forest", "signal": "S2"})

    iforest = IsolationForest(
        n_estimators  = 200,
        contamination = _global_contam,
        max_samples   = "auto",
        random_state  = 42,
        n_jobs        = -1,
    )
    iforest.fit(X_if_tr_pca)

    labels_train = (iforest.predict(X_if_tr_pca) == -1).astype(int)
    labels_val   = (iforest.predict(X_if_va_pca) == -1).astype(int)
    labels_test  = (iforest.predict(X_if_te_pca) == -1).astype(int)

    # Write IF flags back so ensemble flag counts 3 real signals
    import pandas as _pd4
    _if_flags = _pd4.Series(
        dict(list(zip(train_df.index, labels_train)) +
             list(zip(val_df.index,   labels_val))   +
             list(zip(test_df.index,  labels_test))),
        name="signal_isolation_forest")
    df["signal_isolation_forest"] = _if_flags.reindex(df.index).fillna(0).astype(int)

    # Recompute ensemble flag now that IF is real (not placeholder 0)
    df["signal_active_count"] = df[SIGNAL_COLS].sum(axis=1)
    df[TARGET] = (df["signal_active_count"] >= SIGNAL_THRESHOLD).astype(int)
    y_train = train_df.join(df[[TARGET]], rsuffix="_r")[TARGET].values
    y_val   = val_df.join(df[[TARGET]],   rsuffix="_r")[TARGET].values
    y_test  = test_df.join(df[[TARGET]],  rsuffix="_r")[TARGET].values
    y_cv    = np.concatenate([y_train, y_val])
    neg, pos = (y_train==0).sum(), max((y_train==1).sum(), 1)
    SCALE_POS_WEIGHT = round(neg / pos, 2)

    print(f"\nAfter PCA+IF signal update:")
    for s in SIGNAL_COLS:
        print(f"  {s}: {df[s].mean():.2%}")
    print(f"  ensemble_anomaly_flag: {df[TARGET].mean():.2%}")
    print(f"  scale_pos_weight updated: {SCALE_POS_WEIGHT}")

    mlflow.log_params({"n_estimators": 200, "contamination": round(_global_contam, 4),
                       "pca_components": int(_pca.n_components_),
                       "if_features_raw": len(IF_FEATURES)})
    if _dev_contam:
        mlflow.log_params({f"contamination_{k.lower()}": v for k, v in _dev_contam.items()})
    mlflow.log_metric("if_anomaly_rate_test", round(float(labels_test.mean()), 4))
    if len(labels_test) == len(y_test):
        from sklearn.metrics import f1_score as _f1s
        mlflow.log_metric("if_f1_vs_ensemble",
                          round(float(_f1s(y_test, labels_test, zero_division=0)), 4))

    joblib.dump(iforest, "/tmp/ps4_iforest.joblib")
    joblib.dump(scaler,  "/tmp/ps4_if_scaler.joblib")
    joblib.dump(_pca,    "/tmp/ps4_if_pca.joblib")
    mlflow.log_artifact("/tmp/ps4_iforest.joblib",  "isolation_forest")
    mlflow.log_artifact("/tmp/ps4_if_scaler.joblib","isolation_forest")
    mlflow.log_artifact("/tmp/ps4_if_pca.joblib",   "isolation_forest")
    IF_RUN = run.info.run_id

    # IF score as numeric feature (in PCA space → transform back to if_score)
    df_if_score = _pd4.concat([
        _pd4.Series(iforest.decision_function(X_if_tr_pca), index=train_df.index, name="if_score"),
        _pd4.Series(iforest.decision_function(X_if_va_pca), index=val_df.index,   name="if_score"),
        _pd4.Series(iforest.decision_function(X_if_te_pca), index=test_df.index,  name="if_score"),
    ])
    df = df.join(df_if_score)
    if "if_score" not in FEATURE_COLS: FEATURE_COLS.append("if_score")
    X_train = train_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_val   = val_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_test  = test_df.join(df_if_score)[FEATURE_COLS].fillna(0).values
    X_cv    = np.vstack([X_train, X_val])

print(f"IF (PCA-reduced) done. Features: {len(FEATURE_COLS)}")


In [ ]:
# ── CELL 14 : PCA + IF diagnostic charts ────────────────────────────────────
import pandas as _pdv3
from sklearn.metrics import precision_recall_curve, roc_curve, auc as _auc_fn

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── 1. PCA explained variance curve ────────────────────────────────────────
_cum_var = np.cumsum(_pca.explained_variance_ratio_)
axes[0,0].plot(range(1, len(_cum_var)+1), _cum_var, marker="o", ms=4, color="#1565C0")
axes[0,0].axhline(0.95, color="red", ls="--", lw=1, label="95% threshold")
axes[0,0].axvline(_pca.n_components_, color="green", ls="--", lw=1,
                  label=f"{_pca.n_components_} components selected")
axes[0,0].fill_between(range(1, _pca.n_components_+1), _cum_var[:_pca.n_components_],
                       alpha=0.2, color="blue")
axes[0,0].set_title("PCA -- cumulative explained variance", fontsize=10, fontweight="bold")
axes[0,0].set_xlabel("# Principal Components"); axes[0,0].set_ylabel("Variance explained")
axes[0,0].legend(fontsize=8)

# ── 2. IF anomaly score distribution on training set ───────────────────────
_tr_scores = iforest.decision_function(X_if_tr_pca)
axes[0,1].hist(_tr_scores[y_train==0], bins=50, alpha=0.6, color="#1565C0",
               label="Normal (train)", density=True)
axes[0,1].hist(_tr_scores[y_train==1], bins=50, alpha=0.6, color="#E53935",
               label="Anomaly (train)", density=True)
axes[0,1].axvline(0, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[0,1].set_title("IF score distribution (train)", fontsize=10, fontweight="bold")
axes[0,1].set_xlabel("Decision score (lower = more anomalous)")
axes[0,1].legend(fontsize=8)

# ── 3. IF score on test set ─────────────────────────────────────────────────
_te_scores = iforest.decision_function(X_if_te_pca)
axes[0,2].hist(_te_scores[y_test==0], bins=50, alpha=0.6, color="#43A047",
               label="Normal (test)", density=True)
axes[0,2].hist(_te_scores[y_test==1], bins=50, alpha=0.6, color="#FF6F00",
               label="Anomaly (test)", density=True)
axes[0,2].axvline(0, color="black", ls="--", lw=1.2, label="Decision boundary")
axes[0,2].set_title("IF score distribution (test)", fontsize=10, fontweight="bold")
axes[0,2].set_xlabel("Decision score (lower = more anomalous)")
axes[0,2].legend(fontsize=8)

# ── 4. Precision-Recall curve (IF score vs ensemble flag) ──────────────────
_pr_score = -_te_scores   # negate so higher = more anomalous
_prec_c, _rec_c, _ = precision_recall_curve(y_test, _pr_score)
_ap_if = float(np.trapz(_prec_c, _rec_c))
axes[1,0].plot(_rec_c, _prec_c, color="#1565C0", lw=2, label=f"IF (AP={_ap_if:.3f})")
axes[1,0].axhline(y_test.mean(), color="gray", ls="--", lw=1,
                  label=f"Baseline ({y_test.mean():.3f})")
axes[1,0].set_title("Precision-Recall -- IF vs ensemble_anomaly_flag (test)",
                    fontsize=10, fontweight="bold")
axes[1,0].set_xlabel("Recall"); axes[1,0].set_ylabel("Precision")
axes[1,0].legend(fontsize=8)

# ── 5. ROC curve ─────────────────────────────────────────────────────────────
_fpr, _tpr, _ = roc_curve(y_test, _pr_score)
_roc_auc = _auc_fn(_fpr, _tpr)
axes[1,1].plot(_fpr, _tpr, color="#7B1FA2", lw=2, label=f"IF (AUC={_roc_auc:.3f})")
axes[1,1].plot([0,1], [0,1], color="gray", ls="--", lw=1, label="Random")
axes[1,1].set_title("ROC curve -- IF score (test)", fontsize=10, fontweight="bold")
axes[1,1].set_xlabel("False Positive Rate"); axes[1,1].set_ylabel("True Positive Rate")
axes[1,1].legend(fontsize=8)

# ── 6. Per-device contamination bar ────────────────────────────────────────
if _dev_contam:
    _dc = _pdv3.Series(_dev_contam)
    _dc.plot(kind="bar", ax=axes[1,2], color="#00897B", edgecolor="white")
    for bar, v in zip(axes[1,2].patches, _dc.values):
        axes[1,2].text(bar.get_x()+bar.get_width()/2, v+0.001, f"{v:.2%}",
                       ha="center", fontsize=9)
    axes[1,2].set_title("Per-device IF contamination rate", fontsize=10, fontweight="bold")
    axes[1,2].set_ylabel("Contamination"); axes[1,2].tick_params(axis="x", rotation=0)
else:
    axes[1,2].text(0.5, 0.5, "No per-device contamination data",
                   ha="center", va="center", transform=axes[1,2].transAxes)

plt.tight_layout()
plt.savefig("/tmp/ps4_pca_if_charts.png", dpi=100); plt.show()
with mlflow.start_run(run_name="pca_if_diagnostics") as _r:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "visualization"})
    mlflow.log_artifact("/tmp/ps4_pca_if_charts.png")

# ── Top-10 feature loadings table ─────────────────────────────────────────
print("\n-- Top 10 IF features (absolute PCA loadings across first 3 components) --")
_load = _pdv3.DataFrame(
    np.abs(_pca.components_[:3]).T,
    index=IF_FEATURES,
    columns=["PC1", "PC2", "PC3"])
_load["max_load"] = _load.max(axis=1)
print(_load.sort_values("max_load", ascending=False).head(10).round(4).to_string())


In [ ]:
# ── CELL 15 : HDBSCAN — anomaly cluster discovery ──────────────────────────────────────────────────
# Identifies anomaly archetypes in training data.
# Cluster labels + noise flag are added to FEATURE_COLS so supervised models
# (XGBoost / LightGBM / Optuna) can learn which cluster type drives anomalies.
# Val / test rows get approximate cluster assignments via hdbscan.approximate_predict().

with mlflow.start_run(run_name="hdbscan_clustering") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "hdbscan_clustering", "signal": "derived"})

    X_cluster = scaler.transform(train_df[IF_FEATURES].fillna(0).values)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size          = max(50, int(len(X_cluster) * 0.001)),
        min_samples               = 10,
        cluster_selection_epsilon = 0.1,
        metric                    = "euclidean",
        core_dist_n_jobs          = -1,
        prediction_data           = True,   # required for approximate_predict on val/test
    )
    cluster_labels = clusterer.fit_predict(X_cluster)

    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_rate  = (cluster_labels == -1).mean()

    # ── Approximate cluster assignment for val and test rows ──────────────────
    X_cluster_va = scaler.transform(val_df[IF_FEATURES].fillna(0).values)
    X_cluster_te = scaler.transform(test_df[IF_FEATURES].fillna(0).values)
    try:
        labels_va, _ = hdbscan.approximate_predict(clusterer, X_cluster_va)
        labels_te, _ = hdbscan.approximate_predict(clusterer, X_cluster_te)
    except Exception as _ap_err:
        print(f"  [warn] approximate_predict failed ({_ap_err}) — val/test assigned -1")
        import numpy as _np2
        labels_va = _np2.full(len(X_cluster_va), -1, dtype=int)
        labels_te = _np2.full(len(X_cluster_te), -1, dtype=int)

    # ── Write cluster features back to df ─────────────────────────────────────
    import pandas as _pd3, numpy as _np3
    _cl_series = _pd3.concat([
        _pd3.Series(cluster_labels.astype(float), index=train_df.index, name="hdbscan_cluster"),
        _pd3.Series(labels_va.astype(float),      index=val_df.index,   name="hdbscan_cluster"),
        _pd3.Series(labels_te.astype(float),       index=test_df.index,  name="hdbscan_cluster"),
    ])
    _ns_series = (_cl_series == -1).astype(float).rename("hdbscan_noise")
    df["hdbscan_cluster"] = _cl_series.reindex(df.index).fillna(-1)
    df["hdbscan_noise"]   = _ns_series.reindex(df.index).fillna(1)

    for _feat in ["hdbscan_cluster", "hdbscan_noise"]:
        if _feat not in FEATURE_COLS:
            FEATURE_COLS.append(_feat)

    # Rebuild feature arrays with the 2 new cluster columns
    X_train = train_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_val   = val_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_test  = test_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_cv    = _np3.vstack([X_train, X_val])

    # ── MLflow logging ────────────────────────────────────────────────────────
    mlflow.log_params({"min_cluster_size": clusterer.min_cluster_size,
                       "min_samples":      clusterer.min_samples,
                       "metric":           "euclidean"})
    mlflow.log_metrics({"n_clusters":  n_clusters,
                        "noise_rate":  round(noise_rate, 4),
                        "noise_rate_val": round(float((labels_va == -1).mean()), 4),
                        "noise_rate_test": round(float((labels_te == -1).mean()), 4)})

    # Cluster size bar chart
    cluster_sizes = _pd3.Series(cluster_labels).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 4))
    cluster_sizes.iloc[:20].plot(kind="bar", ax=ax, color="#1565C0", edgecolor="white")
    ax.set_title(f"HDBSCAN — {n_clusters} clusters  (noise={noise_rate:.1%})")
    ax.set_xlabel("Cluster ID  (-1 = noise)"); fig.tight_layout()
    fig.savefig("/tmp/hdbscan_clusters.png", dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/hdbscan_clusters.png")

    # Anomaly rate per cluster (shows which archetypes are most anomalous)
    _cl_anom = _pd3.DataFrame({
        "cluster": cluster_labels,
        "anomaly": y_train,
    }).groupby("cluster")["anomaly"].agg(["mean","count"]).rename(
        columns={"mean":"anomaly_rate","count":"n_rows"})
    print(f"HDBSCAN: {n_clusters} clusters  noise_rate={noise_rate:.1%}")
    print(f"\nCluster anomaly rates (top 10 by size):")
    print(_cl_anom.sort_values("n_rows", ascending=False).head(10).to_string())
    print(f"\nFeatures now: {len(FEATURE_COLS)}  (+hdbscan_cluster, +hdbscan_noise)")
    HDBSCAN_RUN = run.info.run_id


In [ ]:
# ── CELL 16 : UMAP + HDBSCAN per device — fault archetype discovery ──────────────────────────────────────────────────
# UMAP: faster than t-SNE, preserves global cluster structure.
# HDBSCAN per device: finds natural fault archetypes without specifying K.
# Results: hdbscan_cluster + hdbscan_noise added to FEATURE_COLS.

try:
    import umap as _umap_lib
    _UMAP_OK = True
except ImportError:
    print("[SKIP] umap-learn not installed — run: pip install umap-learn")
    _UMAP_OK = False

if _UMAP_OK and DEV_COL:
    _all_cl  = {}   # index → cluster label
    _all_ns  = {}   # index → noise flag
    import pandas as _pd5

    for _dt in DEVICE_TYPES:
        _mask_tr = train_df[DEV_COL].str.strip().str.upper() == _dt
        _X_dev   = train_df[_mask_tr][IF_FEATURES].fillna(0).values
        if len(_X_dev) < 100:
            print(f"[SKIP] {_dt}: only {len(_X_dev)} train rows — too few for UMAP")
            for idx in train_df[_mask_tr].index: _all_cl[idx] = -1; _all_ns[idx] = 1
            continue

        print(f"\n{_dt}: UMAP({len(_X_dev):,} rows, {len(IF_FEATURES)} features) ...")
        _X_sc  = scaler.transform(_X_dev)
        _reducer = _umap_lib.UMAP(n_components=10, metric="euclidean",
                                   random_state=42, n_jobs=-1)
        _X_umap  = _reducer.fit_transform(_X_sc)

        _hdb = hdbscan.HDBSCAN(
            min_cluster_size = max(30, int(len(_X_umap) * 0.005)),
            min_samples      = 5,
            metric           = "euclidean",
            core_dist_n_jobs = -1,
            prediction_data  = True,
        )
        _labels = _hdb.fit_predict(_X_umap)
        _n_cl   = len(set(_labels)) - (1 if -1 in _labels else 0)
        _n_rate = (_labels == -1).mean()
        print(f"  {_dt}: {_n_cl} clusters  noise={_n_rate:.1%}")

        # Store train labels
        for idx, lbl in zip(train_df[_mask_tr].index, _labels):
            _all_cl[idx] = float(lbl); _all_ns[idx] = float(lbl == -1)

        # Approximate predict for val + test of same device
        for _split_df, _split_name in [(val_df, "val"), (test_df, "test")]:
            _m_sp = _split_df[DEV_COL].str.strip().str.upper() == _dt
            if _m_sp.sum() == 0: continue
            _X_sp = scaler.transform(_split_df[_m_sp][IF_FEATURES].fillna(0).values)
            _X_sp_umap = _reducer.transform(_X_sp)
            try:
                _lbl_sp, _ = hdbscan.approximate_predict(_hdb, _X_sp_umap)
            except Exception:
                _lbl_sp = _np3.full(len(_X_sp), -1, dtype=int)
            for idx, lbl in zip(_split_df[_m_sp].index, _lbl_sp):
                _all_cl[idx] = float(lbl); _all_ns[idx] = float(lbl == -1)

        # 2D UMAP for visualization
        _viz = _umap_lib.UMAP(n_components=2, metric="euclidean", random_state=42)
        _X2  = _viz.fit_transform(_X_sc)
        _fig, _ax = plt.subplots(figsize=(8, 5))
        _sc = _ax.scatter(_X2[:,0], _X2[:,1], c=_labels, cmap="tab10",
                          s=4, alpha=0.6, rasterized=True)
        _ax.set_title(f"UMAP — {_dt}  ({_n_cl} clusters, noise={_n_rate:.1%})")
        plt.colorbar(_sc, ax=_ax, label="Cluster"); _fig.tight_layout()
        _fig.savefig(f"/tmp/umap_{_dt.lower()}.png", dpi=100); plt.show()
        with mlflow.start_run(run_name=f"umap_hdbscan_{_dt.lower()}") as _run:
            mlflow.set_tags({**COMMON_TAGS, "model_type":"umap_hdbscan",
                             "device_type":_dt})
            mlflow.log_params({"umap_components":10, "hdbscan_min_cluster_size":_hdb.min_cluster_size,
                               "n_clusters":_n_cl, "noise_rate":round(_n_rate,4)})
            mlflow.log_artifact(f"/tmp/umap_{_dt.lower()}.png")

    # Write to df
    import numpy as _np3
    df["hdbscan_cluster"] = _pd5.Series(_all_cl).reindex(df.index).fillna(-1)
    df["hdbscan_noise"]   = _pd5.Series(_all_ns).reindex(df.index).fillna(1)
    for _f in ["hdbscan_cluster","hdbscan_noise"]:
        if _f not in FEATURE_COLS: FEATURE_COLS.append(_f)
    X_train = train_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_val   = val_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_test  = test_df.join(df[["hdbscan_cluster","hdbscan_noise"]])[FEATURE_COLS].fillna(0).values
    X_cv    = np.vstack([X_train, X_val])
    print(f"\nUMAP+HDBSCAN done. Features now: {len(FEATURE_COLS)}")
elif not DEV_COL:
    print("[SKIP] No device column — UMAP per device not possible")


In [ ]:
# ── CELL 17 : Station-level aggregation + paired device supervised model ──────────────────────────────────────────────────
# Station-level: if >50% of devices at a station are anomalous simultaneously
#   → infrastructure fault, not individual device fault.
# Paired device model: wide-format features [dev_A, dev_B, A-B diff]
#   → distinguishes device-level from station-level anomalies.
# Requires: a station/location column in df (STATION_COL auto-detected below).

STATION_COL = next((c for c in df.columns
                    if any(k in c.lower() for k in
                           ["station","location","site","place","stop","facility"])), None)

if STATION_COL is None:
    print(f"[INFO] No station column found in Gold table.")
    print(f"       Columns with 'station/location/site': none detected.")
    print(f"       Add a station→device mapping to Gold ETL to enable this analysis.")
    print(f"       When available, station-level anomaly rate will be computed as:")
    print(f"         station_anomaly_rate = n_anomalous_devices / n_total_devices")
    print(f"         station_fault = station_anomaly_rate > 0.5")
    STATION_ANOMALY_AVAILABLE = False
else:
    print(f"Station column found: {STATION_COL}")
    import pandas as _pd6

    # ── Station-level anomaly rate ─────────────────────────────────────────
    _station_hourly = (df.groupby([STATION_COL, "hour_dt"])
                         .agg(n_devices    = ("DEVICE_ID", "nunique"),
                              n_anomalous  = (TARGET, "sum"))
                         .reset_index())
    _station_hourly["station_anomaly_rate"] = (
        _station_hourly["n_anomalous"] / _station_hourly["n_devices"].clip(lower=1))
    _station_hourly["station_fault"] = (
        _station_hourly["station_anomaly_rate"] > 0.5).astype(int)

    df = df.merge(_station_hourly[[STATION_COL,"hour_dt","station_anomaly_rate","station_fault"]],
                  on=[STATION_COL,"hour_dt"], how="left")
    df["station_anomaly_rate"] = df["station_anomaly_rate"].fillna(0)
    df["station_fault"]        = df["station_fault"].fillna(0).astype(int)

    for _f in ["station_anomaly_rate","station_fault"]:
        if _f not in FEATURE_COLS: FEATURE_COLS.append(_f)

    _sf_rate = _station_hourly["station_fault"].mean()
    print(f"Station-level fault rate: {_sf_rate:.2%}")
    print(f"  (hours where >50% of station devices were anomalous)")

    # ── Paired device supervised model ─────────────────────────────────────
    # Pivot to (station, hour) with per-device metrics side-by-side.
    # Target: station_fault (any device at station anomalous).
    if DEV_COL:
        _wide = (df[[STATION_COL, "hour_dt", DEV_COL, TARGET] + FEATURE_COLS[:10]]
                   .pivot_table(index=[STATION_COL,"hour_dt"],
                                columns=DEV_COL, values=FEATURE_COLS[:10],
                                aggfunc="mean"))
        _wide.columns = [f"{col[0]}_{col[1]}" for col in _wide.columns]
        _wide["station_fault"] = (df.groupby([STATION_COL,"hour_dt"])[TARGET].max()
                                    .reset_index(drop=True))
        _wide = _wide.fillna(0).reset_index(drop=True)
        _Xp   = _wide.drop(columns=["station_fault"]).values
        _yp   = _wide["station_fault"].values

        if len(_Xp) > 200:
            _n_tr = int(len(_Xp) * 0.7)
            _xgb_pair = XGBClassifier(n_estimators=200, max_depth=4,
                                       tree_method="hist", random_state=42,
                                       n_jobs=-1, scale_pos_weight=round((_yp==0).sum()/max((_yp==1).sum(),1),2))
            _xgb_pair.fit(_Xp[:_n_tr], _yp[:_n_tr])
            from sklearn.metrics import roc_auc_score as _ras2
            _pair_auc = _ras2(_yp[_n_tr:], _xgb_pair.predict_proba(_Xp[_n_tr:])[:,1])
            print(f"\nPaired device XGBoost  station_fault AUC={_pair_auc:.4f}")
            with mlflow.start_run(run_name="paired_device_xgb") as _run:
                mlflow.set_tags({**COMMON_TAGS,"model_type":"paired_device_supervised"})
                mlflow.log_params({"n_station_pairs": len(_Xp),
                                   "paired_features": _Xp.shape[1]})
                mlflow.log_metric("station_fault_auc", round(_pair_auc, 4))
        else:
            print("[SKIP] Too few station-hour pairs for paired device model")

    Xtr_new = train_df.join(df[["station_anomaly_rate","station_fault"]])[FEATURE_COLS].fillna(0).values
    Xva_new = val_df.join(df[["station_anomaly_rate","station_fault"]])[FEATURE_COLS].fillna(0).values
    Xte_new = test_df.join(df[["station_anomaly_rate","station_fault"]])[FEATURE_COLS].fillna(0).values
    X_train = Xtr_new; X_val = Xva_new; X_test = Xte_new
    X_cv    = np.vstack([X_train, X_val])
    STATION_ANOMALY_AVAILABLE = True
    print(f"Station features added. Total features: {len(FEATURE_COLS)}")


In [ ]:
# ── CELL 18 : Cross-PS linkage — PS4 anomaly hours → PS3 root cause ──────────────────────────────────────────────────
# When PS4 flags an anomaly hour, what does PS3 predict as the root cause?
# Loads PS3 MLflow run metrics for the same devices.
# Shows: "X% of TVM anomaly hours co-occur with MAJOR/CRITICAL failure days"

PS3_EXPERIMENT = "chicago-ps3-root-cause"

try:
    _ps3_runs = mlflow.search_runs(
        experiment_names=[PS3_EXPERIMENT],
        order_by=["metrics.test_auc_macro DESC"])
    if len(_ps3_runs) == 0:
        print("[INFO] PS3 experiment not found or no runs yet — run PS3 first")
    else:
        # PS3 logs per-device AUC/F1 metrics with keys like tvm_test_auc, gate_test_auc etc.
        _best_ps3 = _ps3_runs.iloc[0]
        print(f"PS3 champion: {_best_ps3.get('tags.mlflow.runName','?')}")

        # Build linkage from df itself if failure_level columns are present
        _fl_col = next((c for c in df.columns
                        if "failure_level" in c.lower() and "label" in c.lower()), None)
        if _fl_col is None:
            # Try joining on date from a PS3 predictions artifact
            print("[INFO] No failure_level column in PS4 Gold.")
            print("       To link: add failure_level_label to PS4 Gold ETL,")
            print("       or join PS4 anomaly scores with PS3 predictions on (DEVICE_ID, date).")

            # Show how the link would look conceptually
            print("\nConceptual cross-PS query:")
            print("  PS4 anomaly hours  JOIN PS3 failure predictions")
            print("  ON: DEVICE_ID AND date(hour_dt) = transit_day")
            print("  Result: P(MAJOR/CRITICAL | PS4_anomaly=1) vs P(MAJOR/CRITICAL | PS4_anomaly=0)")
        else:
            import pandas as _pd7
            _CLASS_NAMES = {0:"MINOR", 1:"MAJOR", 2:"CRITICAL"}
            _link = df[[TARGET, _fl_col]].copy()
            _link["failure_class"] = _link[_fl_col].map(_CLASS_NAMES).fillna("UNKNOWN")
            _crosstab = _pd7.crosstab(_link[TARGET], _link["failure_class"],
                                       normalize="index")
            print(f"\nPS4 anomaly_flag → PS3 failure_level distribution:")
            print(_crosstab.to_string())

            _fig, _ax = plt.subplots(figsize=(8, 4))
            _crosstab.plot(kind="bar", ax=_ax, colormap="RdYlGn_r", edgecolor="white")
            _ax.set_title("PS4 Anomaly Flag × PS3 Failure Level")
            _ax.set_xlabel("PS4 ensemble_anomaly_flag"); _ax.set_ylabel("Proportion")
            _ax.legend(title="Failure level"); _fig.tight_layout()
            _fig.savefig("/tmp/ps4_ps3_linkage.png", dpi=100); plt.show()
            with mlflow.start_run(run_name="cross_ps_linkage") as _run:
                mlflow.set_tags({**COMMON_TAGS, "model_type": "cross_ps_analysis"})
                mlflow.log_artifact("/tmp/ps4_ps3_linkage.png")
except Exception as _ps3_e:
    print(f"[INFO] Cross-PS linkage skipped: {_ps3_e}")


In [ ]:
# ── CELL 19 : Per-device supervised models (TVM / GATE / VALIDATOR) ──────────────────────────────────────────────────
# Trains XGBoost baseline, LightGBM baseline, and Optuna-tuned XGBoost
# for each device type separately. 25 Optuna trials × 3-fold CV for speed.
# Runtime estimate: ~25-35 min per device = 75-105 min total.

PER_DEVICE_RESULTS = {}   # {dev_type: {"best_model","best_run_id","best_auc",...}}

if DEV_COL is None:
    print("[SKIP] No device column — running on ALL devices combined")
    _LOOP_DEVS = [("ALL", df)]
else:
    _LOOP_DEVS = [
        (dt, df[df[DEV_COL].str.strip().str.upper() == dt])
        for dt in DEVICE_TYPES
    ]

for _dev_type, _dev_df in _LOOP_DEVS:
    if len(_dev_df) == 0:
        print(f"[SKIP] {_dev_type}: no data in Gold table")
        continue

    _d_tr = _dev_df[_dev_df["hour_dt"] <  "2025-10-01"]
    _d_va = _dev_df[(_dev_df["hour_dt"] >= "2025-10-01") & (_dev_df["hour_dt"] < "2026-01-01")]
    _d_te = _dev_df[_dev_df["hour_dt"] >= "2026-01-01"]

    if len(_d_va) < 10 or len(_d_te) < 10:
        print(f"[SKIP] {_dev_type}: insufficient val ({len(_d_va)}) or test ({len(_d_te)}) rows")
        continue

    _X_tr = _d_tr[FEATURE_COLS].fillna(0).values;  _y_tr = _d_tr[TARGET].values
    _X_va = _d_va[FEATURE_COLS].fillna(0).values;  _y_va = _d_va[TARGET].values
    _X_te = _d_te[FEATURE_COLS].fillna(0).values;  _y_te = _d_te[TARGET].values
    _X_cv = np.vstack([_X_tr, _X_va]);  _y_cv = np.concatenate([_y_tr, _y_va])
    _neg  = (_y_tr == 0).sum();  _pos = max((_y_tr == 1).sum(), 1)
    _spw  = round(_neg / _pos, 2)

    print(f"\n{'='*62}")
    print(f" {_dev_type}  |  train:{len(_X_tr):,}  val:{len(_X_va):,}  test:{len(_X_te):,}")
    print(f" anomaly rate — train:{_y_tr.mean():.2%}  val:{_y_va.mean():.2%}  test:{_y_te.mean():.2%}")
    print(f"{'='*62}")

    _TAGS     = {**COMMON_TAGS, "device_type": _dev_type}
    _dev_runs = {}   # model_name -> (run_id, test_auc)

    # ── A: XGBoost baseline ───────────────────────────────────────────────────
    with mlflow.start_run(run_name=f"xgb_{_dev_type.lower()}") as _run:
        mlflow.set_tags({**_TAGS, "model_type": "xgboost_supervised"})
        _xgb = XGBClassifier(
            n_estimators=400, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
            scale_pos_weight=_spw, eval_metric="auc", early_stopping_rounds=30,
            tree_method="hist", random_state=42, n_jobs=-1)
        _xgb.fit(_X_tr, _y_tr, eval_set=[(_X_va, _y_va)], verbose=False)
        _vm = evaluate(_xgb, _X_va, _y_va, "val")
        _tm = evaluate(_xgb, _X_te, _y_te, "test")
        mlflow.log_params({"n_estimators": 400, "scale_pos_weight": _spw,
                           "best_iter": _xgb.best_iteration, "device_type": _dev_type})
        mlflow.log_metrics({**_vm, **_tm})
        log_cm(_xgb, _X_te, _y_te, f"xgb_{_dev_type.lower()}")
        _sig = infer_signature(_X_tr, _xgb.predict_proba(_X_tr))
        try: mlflow.xgboost.log_model(_xgb, "model", signature=_sig)
        except Exception as _e: print(f"  [warn] log_model xgb: {_e}")
        _dev_runs["xgb"] = (_run.info.run_id, _tm["test_auc"])
    print(f"  XGB        val={_vm['val_auc']:.4f}  test={_tm['test_auc']:.4f}")

    # ── B: LightGBM baseline ──────────────────────────────────────────────────
    with mlflow.start_run(run_name=f"lgb_{_dev_type.lower()}") as _run:
        mlflow.set_tags({**_TAGS, "model_type": "lightgbm_supervised"})
        _lgb = LGBMClassifier(
            n_estimators=400, num_leaves=63, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=_spw, n_jobs=-1, random_state=42, verbose=-1)
        _lgb.fit(_X_tr, _y_tr, eval_set=[(_X_va, _y_va)],
                 callbacks=[early_stopping(30, verbose=False), log_evaluation(False)])
        _vm = evaluate(_lgb, _X_va, _y_va, "val")
        _tm = evaluate(_lgb, _X_te, _y_te, "test")
        mlflow.log_params({"n_estimators": 400, "best_iter": _lgb.best_iteration_,
                           "device_type": _dev_type})
        mlflow.log_metrics({**_vm, **_tm})
        log_cm(_lgb, _X_te, _y_te, f"lgb_{_dev_type.lower()}")
        _sig = infer_signature(_X_tr, _lgb.predict_proba(_X_tr))
        try: mlflow.lightgbm.log_model(_lgb, "model", signature=_sig)
        except Exception as _e: print(f"  [warn] log_model lgb: {_e}")
        _dev_runs["lgb"] = (_run.info.run_id, _tm["test_auc"])
    print(f"  LGB        val={_vm['val_auc']:.4f}  test={_tm['test_auc']:.4f}")

    # ── C: Optuna XGBoost (25 trials, 3-fold — efficient per-device tuning) ───
    _TS3 = TimeSeriesSplit(n_splits=3)
    def _obj(trial):
        _p = dict(
            n_estimators     = trial.suggest_int("n_estimators", 200, 600),
            max_depth        = trial.suggest_int("max_depth", 3, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            subsample        = trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight = trial.suggest_int("min_child_weight", 1, 20),
            scale_pos_weight = _spw, tree_method="hist", random_state=42, n_jobs=-1)
        _sc = []
        for _st, (_ti, _vi) in enumerate(_TS3.split(_X_cv)):
            _m = XGBClassifier(**_p)
            _m.fit(_X_cv[_ti], _y_cv[_ti],
                   eval_set=[(_X_cv[_vi], _y_cv[_vi])], verbose=False)
            _sc.append(roc_auc_score(_y_cv[_vi], _m.predict_proba(_X_cv[_vi])[:, 1]))
            trial.report(float(np.mean(_sc)), _st)
            if trial.should_prune(): raise optuna.TrialPruned()
        return float(np.mean(_sc))
    _st = optuna.create_study(direction="maximize",
          pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
    _st.optimize(_obj, n_trials=25, show_progress_bar=False)
    _BP = {**_st.best_params, "scale_pos_weight": _spw,
           "tree_method": "hist", "random_state": 42, "n_jobs": -1}
    with mlflow.start_run(run_name=f"xgb_optuna_{_dev_type.lower()}") as _run:
        mlflow.set_tags({**_TAGS, "model_type": "xgboost_optuna_tuned"})
        mlflow.log_params({**_BP, "optuna_trials": 25, "cv_folds": 3,
                           "optuna_cv": round(_st.best_value, 4), "device_type": _dev_type})
        _xgb_t = XGBClassifier(**_BP)
        _xgb_t.fit(_X_cv, _y_cv, verbose=False)
        _vm = evaluate(_xgb_t, _X_va, _y_va, "val")
        _tm = evaluate(_xgb_t, _X_te, _y_te, "test")
        mlflow.log_metrics({**_vm, **_tm})
        log_cm(_xgb_t, _X_te, _y_te, f"xgb_optuna_{_dev_type.lower()}")
        _sig = infer_signature(_X_tr, _xgb_t.predict_proba(_X_tr))
        try: mlflow.xgboost.log_model(_xgb_t, "model", signature=_sig)
        except Exception as _e: print(f"  [warn] log_model optuna: {_e}")
        _dev_runs["xgb_optuna"] = (_run.info.run_id, _tm["test_auc"])
    print(f"  XGB(Optuna) val={_vm['val_auc']:.4f}  test={_tm['test_auc']:.4f}")

    # ── D: Signal voting (per-device evaluation) ──────────────────────────────
    _avail_sigs = [c for c in SIGNAL_COLS if c in _d_te.columns]
    if _avail_sigs:
        class _SVE:
            def __init__(self, sc, thr): self.signal_cols=sc; self.threshold=thr
            def predict_proba(self, Xdf):
                act=Xdf[self.signal_cols].sum(axis=1).values
                p1=np.clip(act/max(len(self.signal_cols),1),0,1)
                return np.column_stack([1-p1,p1])
            def predict(self, Xdf):
                return (Xdf[self.signal_cols].sum(axis=1)>=self.threshold).astype(int).values
        _voter = _SVE(_avail_sigs, SIGNAL_THRESHOLD)
        _vm_v  = evaluate(_voter, _d_va, _y_va, "val")
        _tm_v  = evaluate(_voter, _d_te, _y_te, "test")
        with mlflow.start_run(run_name=f"signal_vote_{_dev_type.lower()}") as _run:
            mlflow.set_tags({**_TAGS, "model_type": "signal_voting_deterministic"})
            mlflow.log_params({"signals": str(_avail_sigs), "threshold": SIGNAL_THRESHOLD,
                               "device_type": _dev_type})
            mlflow.log_metrics({**_vm_v, **_tm_v})
        print(f"  Voting     val={_vm_v['val_auc']:.4f}  test={_tm_v['test_auc']:.4f}")

    # ── SHAP for best model of this device ────────────────────────────────────
    _best_nm, (_best_rid, _best_auc) = max(_dev_runs.items(), key=lambda x: x[1][1])
    _best_mdl = _xgb_t if _best_nm == "xgb_optuna" else (_xgb if _best_nm == "xgb" else _lgb)
    try:
        _X_shap = pd.DataFrame(_X_te[:min(800, len(_X_te))], columns=FEATURE_COLS)
        _expl   = shap.TreeExplainer(_best_mdl)
        _sv     = _expl(_X_shap)
        shap.plots.bar(_sv, max_display=15, show=False)
        _fig = plt.gcf(); _fig.tight_layout()
        _fig.savefig(f"/tmp/ps4_shap_{_dev_type.lower()}.png", dpi=100); plt.close(_fig)
        with mlflow.start_run(run_name=f"shap_{_dev_type.lower()}") as _run:
            mlflow.set_tags({**_TAGS, "model_type": "shap_analysis"})
            mlflow.log_artifact(f"/tmp/ps4_shap_{_dev_type.lower()}.png")
    except Exception as _se:
        print(f"  [warn] SHAP {_dev_type}: {_se}")

    PER_DEVICE_RESULTS[_dev_type] = {
        "best_model"  : _best_nm,
        "best_run_id" : _best_rid,
        "best_auc"    : _best_auc,
        "runs"        : _dev_runs,
        "n_train"     : len(_X_tr),
        "n_val"       : len(_X_va),
        "n_test"      : len(_X_te),
        "best_mdl_obj": _best_mdl,
    }
    print(f"\n  ★ {_dev_type} champion: {_best_nm}  test_auc={_best_auc:.4f}")

print(f"\n\n{'='*62}")
print(f"{'DEVICE':12s} {'CHAMPION':22s} {'TEST AUC':>9}  {'TRAIN N':>9}  {'TEST N':>7}")
print("-"*62)
for _dt, _r in PER_DEVICE_RESULTS.items():
    print(f"{_dt:12s} {_r['best_model']:22s} {_r['best_auc']:9.4f}  {_r['n_train']:9,}  {_r['n_test']:7,}")


In [ ]:
# ── CELL 20 : Per-device model evaluation charts ──────────────────────────────────────────────────
import pandas as _pdv4
from sklearn.metrics import precision_recall_curve, roc_curve, auc as _auc_fn2

if PER_DEVICE_RESULTS:
    n_dev = len(PER_DEVICE_RESULTS)
    fig, axes = plt.subplots(3, n_dev, figsize=(7*n_dev, 15))
    if n_dev == 1: axes = axes.reshape(3, 1)

    _summary_rows = []
    for col_i, (_dt, _res) in enumerate(PER_DEVICE_RESULTS.items()):
        _yt    = _res.get("y_test")
        _prob  = _res.get("best_prob_test")
        _name  = _res.get("best_model_name", _dt)
        _fimp  = _res.get("feature_importances")

        if _yt is None or _prob is None:
            for r in range(3): axes[r, col_i].set_visible(False)
            continue

        # ── Row 0: ROC curve ────────────────────────────────────────────────
        _fpr, _tpr, _ = roc_curve(_yt, _prob)
        _rauc = _auc_fn2(_fpr, _tpr)
        axes[0, col_i].plot(_fpr, _tpr, color="#1565C0", lw=2,
                            label=f"{_name} (AUC={_rauc:.3f})")
        axes[0, col_i].plot([0,1],[0,1],"--",color="gray",lw=1,label="Random")
        axes[0, col_i].set_title(f"{_dt} — ROC curve", fontsize=11, fontweight="bold")
        axes[0, col_i].set_xlabel("FPR"); axes[0, col_i].set_ylabel("TPR")
        axes[0, col_i].legend(fontsize=8)

        # ── Row 1: PR curve ─────────────────────────────────────────────────
        _prec, _rec, _ = precision_recall_curve(_yt, _prob)
        _ap = float(np.trapz(_prec, _rec))
        axes[1, col_i].plot(_rec, _prec, color="#7B1FA2", lw=2,
                            label=f"{_name} (AP={_ap:.3f})")
        axes[1, col_i].axhline(_yt.mean(), color="gray", ls="--", lw=1,
                               label=f"Baseline ({_yt.mean():.3f})")
        axes[1, col_i].set_title(f"{_dt} — Precision-Recall", fontsize=11, fontweight="bold")
        axes[1, col_i].set_xlabel("Recall"); axes[1, col_i].set_ylabel("Precision")
        axes[1, col_i].legend(fontsize=8)

        # ── Row 2: Top-15 feature importance ───────────────────────────────
        if _fimp is not None and len(_fimp) == len(FEATURE_COLS):
            _fi = _pdv4.Series(_fimp, index=FEATURE_COLS).nlargest(15)
            _fi.sort_values().plot(kind="barh", ax=axes[2, col_i],
                                   color="#00897B", edgecolor="white")
            axes[2, col_i].set_title(f"{_dt} — Top 15 feature importance",
                                     fontsize=11, fontweight="bold")
            axes[2, col_i].set_xlabel("Importance")
        else:
            axes[2, col_i].text(0.5, 0.5, "No feature importance", ha="center",
                                va="center", transform=axes[2, col_i].transAxes)

        # Collect summary row
        _summary_rows.append({
            "Device": _dt, "Best_Model": _name,
            "Test_AUC": round(_rauc, 4), "Test_AP": round(_ap, 4),
            "Test_Anomaly_Rate": f"{_yt.mean():.2%}",
            "Train_N": _res.get("train_n", "?"), "Test_N": _res.get("test_n", "?"),
        })

    plt.tight_layout()
    plt.savefig("/tmp/ps4_perdev_evaluation.png", dpi=100); plt.show()
    with mlflow.start_run(run_name="per_device_evaluation_charts") as _r:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"visualization"})
        mlflow.log_artifact("/tmp/ps4_perdev_evaluation.png")

    # ── Summary table ─────────────────────────────────────────────────────
    if _summary_rows:
        print("\n── Per-Device Champion Summary ──")
        print(_pdv4.DataFrame(_summary_rows).to_string(index=False))

    # ── Model comparison table: all models per device ─────────────────────
    print("\n── Model metric comparison (all trained models) ──")
    for _dt, _res in PER_DEVICE_RESULTS.items():
        _mc = _res.get("model_comparison")
        if _mc is not None:
            print(f"\n  {_dt}:")
            print("  " + _pdv4.DataFrame(_mc).to_string(index=False).replace("\n","\n  "))
else:
    print("[INFO] PER_DEVICE_RESULTS is empty — run Cell 16 first.")


In [ ]:
# ── CELL 21 : Stacking ensemble (combined, all devices) ──────────────────────────────────────────────────
# Trains a single stacking ensemble on all data for a global baseline.
# Per-device champions from Cell 13 are preferred for production scoring.

with mlflow.start_run(run_name="stacking_ensemble_all") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type": "stacking_ensemble", "device_type": "ALL"})
    base = [
        ("xgb", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                               scale_pos_weight=SCALE_POS_WEIGHT,
                               tree_method="hist", random_state=42, n_jobs=-1, verbosity=0)),
        ("lgb", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                               scale_pos_weight=SCALE_POS_WEIGHT,
                               n_jobs=-1, random_state=42, verbose=-1)),
        ("rf",  RandomForestClassifier(n_estimators=150, max_depth=10,
                                        class_weight="balanced_subsample",
                                        n_jobs=-1, random_state=42)),
    ]
    stack = StackingClassifier(
        estimators=base,
        final_estimator=CalibratedClassifierCV(
            LogisticRegression(C=1.0, max_iter=500), cv=3, method="isotonic"),
        cv=TimeSeriesSplit(n_splits=3), stack_method="predict_proba",
        passthrough=True, n_jobs=-1)
    stack.fit(X_cv, y_cv)
    vm = evaluate(stack, X_val, y_val, "val")
    tm = evaluate(stack, X_test, y_test, "test")
    mlflow.log_metrics({**vm, **tm})
    log_cm(stack, X_test, y_test, "stack_all")
    sig = infer_signature(X_train, stack.predict_proba(X_train))
    try: mlflow.sklearn.log_model(stack, "model", signature=sig)
    except Exception as _e: print(f"  [warn] log_model stack: {_e}")
    STACK_RUN = run.info.run_id
print(f"Stack(all)  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")


In [ ]:
# ── CELL 22 : Stacking ensemble diagnostic charts ──────────────────────────────────────────────────
import pandas as _pdv5
from sklearn.metrics import (precision_recall_curve, roc_curve, auc as _auc_fn3,
                              brier_score_loss, confusion_matrix as _cm_fn)
from sklearn.calibration import calibration_curve

_s_prob_val  = stack.predict_proba(X_val)[:, 1]
_s_prob_test = stack.predict_proba(X_test)[:, 1]
_s_pred_test = (_s_prob_test >= 0.5).astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ── 1. ROC curve ──────────────────────────────────────────────────────────
_fpr, _tpr, _ = roc_curve(y_test, _s_prob_test)
_rauc = _auc_fn3(_fpr, _tpr)
axes[0,0].plot(_fpr, _tpr, lw=2, color="#1565C0", label=f"Stack (AUC={_rauc:.3f})")
axes[0,0].plot([0,1],[0,1],"--",color="gray",lw=1)
axes[0,0].set_title("ROC Curve — Stacking Ensemble (test)", fontsize=10, fontweight="bold")
axes[0,0].set_xlabel("False Positive Rate"); axes[0,0].set_ylabel("True Positive Rate")
axes[0,0].legend(fontsize=9)

# ── 2. Precision-Recall curve ──────────────────────────────────────────────
_prec, _rec, _thr = precision_recall_curve(y_test, _s_prob_test)
_ap = float(np.trapz(_prec, _rec))
axes[0,1].plot(_rec, _prec, lw=2, color="#7B1FA2", label=f"Stack (AP={_ap:.3f})")
axes[0,1].axhline(y_test.mean(), color="gray", ls="--", lw=1,
                  label=f"Baseline ({y_test.mean():.3f})")
axes[0,1].set_title("Precision-Recall Curve (test)", fontsize=10, fontweight="bold")
axes[0,1].set_xlabel("Recall"); axes[0,1].set_ylabel("Precision")
axes[0,1].legend(fontsize=9)

# ── 3. Calibration curve (reliability diagram) ─────────────────────────────
_frac_pos, _mean_pred = calibration_curve(y_test, _s_prob_test, n_bins=10)
axes[0,2].plot(_mean_pred, _frac_pos, "s-", color="#00897B", lw=2, label="Stack")
axes[0,2].plot([0,1],[0,1],"--",color="gray",lw=1,label="Perfect calibration")
axes[0,2].set_title("Calibration curve (reliability diagram, test)",
                    fontsize=10, fontweight="bold")
axes[0,2].set_xlabel("Mean predicted probability")
axes[0,2].set_ylabel("Fraction of positives")
axes[0,2].legend(fontsize=9)

# ── 4. Confusion matrix ───────────────────────────────────────────────────
_cm = _cm_fn(y_test, _s_pred_test)
sns.heatmap(_cm, annot=True, fmt="d", cmap="Blues", ax=axes[1,0],
            xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"])
axes[1,0].set_title("Confusion Matrix (threshold=0.5, test)", fontsize=10, fontweight="bold")
axes[1,0].set_xlabel("Predicted"); axes[1,0].set_ylabel("Actual")

# ── 5. Predicted probability histogram ─────────────────────────────────────
axes[1,1].hist(_s_prob_test[y_test==0], bins=50, alpha=0.6, color="#1565C0",
               density=True, label="Normal")
axes[1,1].hist(_s_prob_test[y_test==1], bins=50, alpha=0.6, color="#E53935",
               density=True, label="Anomaly")
axes[1,1].axvline(0.5, color="black", ls="--", lw=1.2, label="Threshold=0.5")
axes[1,1].set_title("Predicted probability distribution (test)", fontsize=10, fontweight="bold")
axes[1,1].set_xlabel("P(anomaly)"); axes[1,1].legend(fontsize=9)

# ── 6. Precision / Recall / F1 at various thresholds ───────────────────────
_thrs = np.linspace(0.05, 0.95, 30)
_precs, _recs, _f1s = [], [], []
for _t in _thrs:
    _p_t = (_s_prob_test >= _t).astype(int)
    from sklearn.metrics import precision_score as _ps2, recall_score as _rs2, f1_score as _f1s2
    _precs.append(_ps2(y_test, _p_t, zero_division=0))
    _recs.append(_rs2(y_test, _p_t, zero_division=0))
    _f1s.append(_f1s2(y_test, _p_t, zero_division=0))
axes[1,2].plot(_thrs, _precs, label="Precision", color="#1565C0", lw=2)
axes[1,2].plot(_thrs, _recs,  label="Recall",    color="#E53935",  lw=2)
axes[1,2].plot(_thrs, _f1s,   label="F1",        color="#43A047",  lw=2, ls="--")
_best_t = _thrs[int(np.argmax(_f1s))]
axes[1,2].axvline(_best_t, color="gray", ls=":", lw=1.5,
                  label=f"Best F1 threshold={_best_t:.2f}")
axes[1,2].set_title("Precision / Recall / F1 vs threshold (test)",
                    fontsize=10, fontweight="bold")
axes[1,2].set_xlabel("Classification threshold"); axes[1,2].legend(fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/ps4_stack_charts.png", dpi=100); plt.show()
with mlflow.start_run(run_name="stacking_ensemble_charts") as _r:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"visualization"})
    mlflow.log_artifact("/tmp/ps4_stack_charts.png")

# ── Metric table ──────────────────────────────────────────────────────────
print("\n── Stacking Ensemble Metrics ──")
_brier = brier_score_loss(y_test, _s_prob_test)
import pandas as _pdv5
print(_pdv5.DataFrame([{
    "Split":"Val",  "AUC": round(_auc_fn3(*roc_curve(y_val, _s_prob_val)[:2], 0), 4),
    "AP": round(float(np.trapz(*precision_recall_curve(y_val, _s_prob_val)[:2])), 4),
    "Brier": round(brier_score_loss(y_val, _s_prob_val), 4),
}, {
    "Split":"Test", "AUC": round(_rauc, 4), "AP": round(_ap, 4),
    "Brier": round(_brier, 4),
    "Best_F1_Thr": round(_best_t, 3),
    "F1@0.5": round(float(max(_f1s)), 4),
}]).to_string(index=False))


In [ ]:
# ── CELL 23 : Select champion per device + register ──────────────────────────────────────────────────
# Per-device: each device type gets its own MLflow model registration.
# Global fallback: stacking ensemble registered separately.

mlfc = mlflow.tracking.MlflowClient()

# Register per-device champions
CHAMPION_RUN_IDS = {}   # {dev_type: run_id}
for _dt, _res in PER_DEVICE_RESULTS.items():
    _mdl_name = f"{MLFLOW_MODEL_NAME}-{_dt.lower()}"
    try: mlfc.create_registered_model(_mdl_name,
            description=f"PS4 Anomaly Detection — {_dt} — Chicago Ventra")
    except: pass
    _mv = mlfc.create_model_version(
        name=_mdl_name,
        source=f"runs:/{_res['best_run_id']}/model",
        run_id=_res["best_run_id"],
        tags={"algorithm": _res["best_model"], "test_auc": str(_res["best_auc"]),
              "device_type": _dt, "gold_version": str(GOLD_VERSION)})
    mlfc.transition_model_version_stage(_mdl_name, _mv.version, "Staging")
    CHAMPION_RUN_IDS[_dt] = _res["best_run_id"]
    print(f"MLflow: {_mdl_name} v{_mv.version} → Staging  (AUC={_res['best_auc']:.4f})")

# Overall champion (highest test AUC across all device types)
if PER_DEVICE_RESULTS:
    CHAMPION_DEVICE = max(PER_DEVICE_RESULTS, key=lambda k: PER_DEVICE_RESULTS[k]["best_auc"])
    CHAMPION_RUN_ID = PER_DEVICE_RESULTS[CHAMPION_DEVICE]["best_run_id"]
    CHAMPION_NAME   = PER_DEVICE_RESULTS[CHAMPION_DEVICE]["best_model"]
    CHAMPION_AUC    = PER_DEVICE_RESULTS[CHAMPION_DEVICE]["best_auc"]
    MLFLOW_VERSION  = "per-device"
    print(f"\nOverall champion: {CHAMPION_DEVICE}/{CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}")
else:
    CHAMPION_RUN_ID = None; CHAMPION_NAME = "none"; CHAMPION_AUC = -1.0; MLFLOW_VERSION = 0
    print("[WARN] No per-device results — check DEV_COL detection in Cell 5")


In [ ]:
# ── CELL 24 : Deploy champion endpoint + monitoring ──────────────────────────────────────────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor, DataCaptureConfig,
                                      CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

# Clean up prior endpoint + config
for _ec_name in [ENDPOINT_NAME]:
    try:
        SM_CLIENT.delete_endpoint(EndpointName=_ec_name)
        print(f"Deleted existing endpoint: {_ec_name}")
        import time as _t; _t.sleep(10)
    except Exception: pass
    try:
        SM_CLIENT.delete_endpoint_config(EndpointConfigName=_ec_name)
        print(f"Deleted existing endpoint config: {_ec_name}")
    except Exception: pass

predictor = None
if CHAMPION_RUN_ID is None:
    print("[SKIP] No champion run — skipping deploy.")
else:
    try:
        import sagemaker.image_uris as _sm_img
        _image = _sm_img.retrieve(framework="sklearn", region=REGION, version="1.2-1",
                                   instance_type="ml.m5.large", image_scope="inference")
        _champ_run  = mlfc.get_run(CHAMPION_RUN_ID)
        _champ_type = _champ_run.data.tags.get("model_type", "")
        if "xgboost" in _champ_type:
            champion_model = mlflow.xgboost.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
        elif "lightgbm" in _champ_type:
            champion_model = mlflow.lightgbm.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
        else:
            champion_model = mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
        joblib.dump(champion_model, "/tmp/ps4_champion.joblib")
        joblib.dump(FEATURE_COLS,   "/tmp/ps4_feature_cols.joblib")
        import tarfile as _tf
        with _tf.open("/tmp/model.tar.gz", "w:gz") as _tar:
            _tar.add("/tmp/ps4_champion.joblib", arcname="ps4_champion.joblib")
            _tar.add("/tmp/ps4_feature_cols.joblib", arcname="ps4_feature_cols.joblib")
        _MODEL_S3_KEY = f"sagemaker/ps4/model-v1/model.tar.gz"
        S3_CLIENT.upload_file("/tmp/model.tar.gz", ARTIFACT_BUCKET, _MODEL_S3_KEY)
        try: SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
                ModelPackageGroupDescription="PS4 Anomaly Detection — Chicago Ventra")
        except: pass
        resp = SM_CLIENT.create_model_package(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageDescription=f"{CHAMPION_NAME}|AUC={CHAMPION_AUC:.4f}",
            InferenceSpecification={
                "Containers": [{"Image": _image,
                                "ModelDataUrl": f"s3://{ARTIFACT_BUCKET}/{_MODEL_S3_KEY}"}],
                "SupportedContentTypes": ["text/csv"],
                "SupportedResponseMIMETypes": ["application/json"],
                "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large"]},
            ModelApprovalStatus="PendingManualApproval",
            CustomerMetadataProperties={"ps": "PS4", "algorithm": CHAMPION_NAME,
                "test_auc": str(round(CHAMPION_AUC, 4)), "device_type": CHAMPION_DEVICE})
        SM_MODEL_PKG_ARN = resp["ModelPackageArn"]
        SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                        ModelApprovalStatus="Approved")
        _pkg = ModelPackage(role=ROLE, model_package_arn=SM_MODEL_PKG_ARN,
                             sagemaker_session=SM_SESSION)
        predictor = _pkg.deploy(
            initial_instance_count=1, instance_type="ml.m5.large",
            endpoint_name=ENDPOINT_NAME,
            data_capture_config=DataCaptureConfig(
                enable_capture=True, sampling_percentage=100,
                destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/data-capture/",
                capture_options=["Input", "Output"]))
        print(f"Endpoint deployed: {ENDPOINT_NAME}")
    except Exception as _de:
        print(f"[WARN] Deploy failed: {_de}")

if predictor is not None:
    monitor = DefaultModelMonitor(role=ROLE, instance_count=1,
        instance_type="ml.m5.large", sagemaker_session=SM_SESSION)
    pd.DataFrame(X_train, columns=FEATURE_COLS).to_csv("/tmp/ps4_baseline.csv", index=False)
    S3_CLIENT.upload_file("/tmp/ps4_baseline.csv", ARTIFACT_BUCKET,
                          "sagemaker/ps4/baseline/baseline.csv")
    monitor.suggest_baseline(
        baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/baseline/baseline.csv",
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/baseline/output/",
        wait=True, logs=False)
    monitor.create_monitoring_schedule(
        monitor_schedule_name="chicago-ps4-data-quality",
        endpoint_input=ENDPOINT_NAME,
        output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps4/monitor-output/",
        statistics=monitor.baseline_statistics(),
        constraints=monitor.suggested_constraints(),
        schedule_cron_expression="cron(0 */6 * * ? *)",
        enable_cloudwatch_metrics=True)
    print("PS4 monitor: every 6 hours")
    cw = boto3.client("cloudwatch", region_name=REGION)
    cw.put_metric_alarm(
        AlarmName="PS4-DataQuality-ViolationRate",
        ComparisonOperator="GreaterThanThreshold", EvaluationPeriods=1,
        MetricName="feature_baseline_drift_percentage",
        Namespace="aws/sagemaker/Endpoints/data-metrics",
        Period=21600, Statistic="Average", Threshold=3.0,
        AlarmDescription="PS4 anomaly input violation >3% — CUBIC MARS 97% gate",
        Dimensions=[{"Name": "Endpoint", "Value": ENDPOINT_NAME}],
        TreatMissingData="notBreaching")
    print("CloudWatch alarm set.")
else:
    print("[SKIP] Monitor not scheduled — endpoint not deployed.")


In [ ]:
# ── CELL 25 : Summary ──────────────────────────────────────────────────
print("="*62)
print("PS4 — Hourly Anomaly Detection — Complete")
print("="*62)
print(f"Device column : {DEV_COL}")
print(f"Signal cols   : {SIGNAL_COLS}")
print(f"Monitor freq  : every 6h (hourly grain)")
print()
print(f"{'DEVICE':12s} {'CHAMPION':22s} {'AUC':>8}  {'TRAIN':>8}  {'VAL':>6}  {'TEST':>6}")
print("-"*62)
for _dt, _r in PER_DEVICE_RESULTS.items():
    print(f"{_dt:12s} {_r['best_model']:22s} {_r['best_auc']:8.4f}  "
          f"{_r['n_train']:8,}  {_r['n_val']:6,}  {_r['n_test']:6,}")
print()
print(f"Signal rates (global):")
for s in SIGNAL_COLS:
    rate = df[s].mean() if s in df.columns else 0
    print(f"  {s:35s}: {rate:.2%}")
print()
if PER_DEVICE_RESULTS:
    print(f"MLflow experiment : {EXPERIMENT_NAME}")
    for _dt in PER_DEVICE_RESULTS:
        print(f"  Registered model: {MLFLOW_MODEL_NAME}-{_dt.lower()}")


In [ ]:
# ── CELL 26 : Summary dashboard ──────────────────────────────────────────────────
import pandas as _pdv6

print("=" * 70)
print("PS4 — Hourly Anomaly Detection — Run Summary")
print("=" * 70)

# ── 1. Signal health table ─────────────────────────────────────────────────
print("\n[1] Signal Health")
_rows = [{"Signal": s,
          "Rate": f"{df[s].mean():.2%}",
          "Status": "ACTIVE" if df[s].mean() > 0.001 else "DORMANT"}
         for s in SIGNAL_COLS + [TARGET] if s in df.columns]
print(_pdv6.DataFrame(_rows).to_string(index=False))

# ── 2. Feature dimension table ─────────────────────────────────────────────
print(f"\n[2] Feature Space")
print(f"   Raw features in FEATURE_COLS : {len(FEATURE_COLS)}")
if "_pca" in dir():
    print(f"   IF features (raw)            : {len(IF_FEATURES)}")
    print(f"   PCA components (95% var)     : {_pca.n_components_}")

# ── 3. Train / Val / Test size ─────────────────────────────────────────────
print(f"\n[3] Data splits")
_sp = _pdv6.DataFrame([
    {"Split":"Train","Rows":len(X_train),"Anomaly_Rate":f"{y_train.mean():.2%}",
     "Date_Range":f"{train_df['hour_dt'].min().date()} → {train_df['hour_dt'].max().date()}"},
    {"Split":"Val",  "Rows":len(X_val),  "Anomaly_Rate":f"{y_val.mean():.2%}",
     "Date_Range":f"{val_df['hour_dt'].min().date()} → {val_df['hour_dt'].max().date()}"},
    {"Split":"Test", "Rows":len(X_test), "Anomaly_Rate":f"{y_test.mean():.2%}",
     "Date_Range":f"{test_df['hour_dt'].min().date()} → {test_df['hour_dt'].max().date()}"},
])
print(_sp.to_string(index=False))

# ── 4. Per-device champion table ──────────────────────────────────────────
if "PER_DEVICE_RESULTS" in dir() and PER_DEVICE_RESULTS:
    print("\n[4] Per-Device Champions")
    _champ_rows = []
    for _dt, _r in PER_DEVICE_RESULTS.items():
        _champ_rows.append({
            "Device": _dt,
            "Champion": _r.get("best_model_name", "?"),
            "Test_AUC": _r.get("test_auc", "?"),
            "Test_AP":  _r.get("test_ap",  "?"),
            "scale_pos_weight": _r.get("spw", "?"),
        })
    print(_pdv6.DataFrame(_champ_rows).to_string(index=False))

# ── 5. Dormant signal warning ─────────────────────────────────────────────
_dormant = [s for s in SIGNAL_COLS if s in df.columns and df[s].mean() < 0.001]
if _dormant:
    print(f"\n[WARN] Dormant signals (0% activation — check Gold ETL columns):")
    for s in _dormant:
        print(f"   {s}")

print("\n" + "=" * 70)


In [ ]:
# ── CELL 27 : Anomaly rate forecasting — Prophet ──────────────────────────────────────────────────
# Forecasts daily anomaly rate per device type for the next 7 days.
# Useful for maintenance scheduling: "GATE anomaly rate expected to peak Thu-Fri".
# Prophet decomposes trend + weekly seasonality + holiday effects.

try:
    from prophet import Prophet as _Prophet
    import pandas as _pd8
    _PROPHET_OK = True
except ImportError:
    print("[SKIP] Prophet not installed — run: pip install prophet")
    _PROPHET_OK = False

if _PROPHET_OK:
    _forecast_results = {}
    _dev_groups = [(dt, df[df[DEV_COL].str.strip().str.upper() == dt])
                   if DEV_COL else ("ALL", df)
                   for dt in (DEVICE_TYPES if DEV_COL else ["ALL"])]

    _fig, _axes = plt.subplots(len(_dev_groups), 1,
                                figsize=(12, 4 * len(_dev_groups)))
    if len(_dev_groups) == 1: _axes = [_axes]

    for _ax, (_dt, _ddf) in zip(_axes, _dev_groups):
        if len(_ddf) < 48:   # need at least 2 days of hourly data
            _ax.set_title(f"{_dt}: insufficient data"); continue

        # Daily anomaly rate time series
        _daily = (_ddf.assign(ds=_ddf["hour_dt"].dt.floor("D"))
                     .groupby("ds")[TARGET].mean()
                     .reset_index()
                     .rename(columns={TARGET: "y"}))
        _daily = _daily[_daily["y"].notna()]

        _m = _Prophet(yearly_seasonality=True, weekly_seasonality=True,
                      daily_seasonality=False, changepoint_prior_scale=0.05)
        _m.fit(_daily)

        _future   = _m.make_future_dataframe(periods=7)
        _forecast = _m.predict(_future)
        _forecast_results[_dt] = _forecast[["ds","yhat","yhat_lower","yhat_upper"]].tail(14)

        # Plot
        _hist = _daily.tail(60)
        _pred = _forecast.tail(7)
        _ax.plot(_hist["ds"], _hist["y"], "b-", lw=1.5, label="Historical")
        _ax.plot(_pred["ds"], _pred["yhat"], "r--", lw=2, label="Forecast")
        _ax.fill_between(_pred["ds"], _pred["yhat_lower"], _pred["yhat_upper"],
                         alpha=0.2, color="red", label="95% CI")
        _ax.set_title(f"{_dt} — 7-day anomaly rate forecast")
        _ax.set_ylabel("Anomaly rate"); _ax.legend(fontsize=8)
        _ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

        with mlflow.start_run(run_name=f"prophet_forecast_{_dt.lower()}") as _run:
            mlflow.set_tags({**COMMON_TAGS, "model_type":"prophet_forecast",
                             "device_type":_dt})
            mlflow.log_param("forecast_horizon_days", 7)
            mlflow.log_metric("mean_daily_anomaly_rate",
                              round(float(_daily["y"].mean()), 4))

    plt.tight_layout()
    plt.savefig("/tmp/ps4_forecast.png", dpi=100); plt.show()
    with mlflow.start_run(run_name="anomaly_forecast_summary") as _run:
        mlflow.set_tags({**COMMON_TAGS, "model_type":"prophet_forecast"})
        mlflow.log_artifact("/tmp/ps4_forecast.png")
    print("Prophet forecasting done.")


In [ ]:
# ── FINAL CELL : Export notebook with all outputs to HTML ───────────────────
# Permanent fix for SageMaker Studio output stripping.
# Run this once at the end of a full notebook run.
# The HTML file is saved to EFS (/home/sagemaker-user/PS4/) and persists
# across sessions, instance stops, and kernel restarts.
# Open it in any browser to see all cell outputs, tables, and charts.

import subprocess, os, datetime

_nb_path  = f"/home/sagemaker-user/PS4/PS4.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"/home/sagemaker-user/PS4/PS4_output_{_ts}.html"

if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True
    )
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved → {_out_html}  ({_size:.0f} KB)")
        print(f"  Open in browser — all cell outputs preserved permanently.")
    else:
        print(f"[HTML EXPORT] nbconvert error:")
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
        # Fallback: save key outputs to a text log instead
        _log = f"/home/sagemaker-user/PS4/PS4_run_{_ts}.log"
        print(f"  Fallback: write key results to {_log} manually.")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Actual path on EFS may differ — update _nb_path above.")
    print(f"  Run: !find /home/sagemaker-user -name 'PS4.ipynb' to locate it.")
